## Merge and Extraction of Data

### Convert Corruption Index

In [12]:
import pandas as pd
import os
os.makedirs("Datasets/Corruption Index", exist_ok=True)

# Path to your raw text file
input_file = "Datasets/Corruption Index/cpi2024_raw.txt"
output_file = "Datasets/Corruption Index/cpi2024.csv"

# Read as tab-separated text
df = pd.read_csv(input_file, sep="\t", engine="python")

# Save as CSV
df.to_csv(output_file, index=False, encoding="utf-8")

print("CSV created successfully:", output_file)


CSV created successfully: Datasets/Corruption Index/cpi2024.csv


### Combine Education Expenditure

In [21]:
import pandas as pd
import re

def merge_two_panels(
    path1,
    path2,
    country_col1,
    country_col2,
    join_mode="union",
    output_path=None
):
    # Read with correct encoding
    df1 = pd.read_csv(path1, encoding="cp1252")
    df2 = pd.read_csv(path2, encoding="cp1252")
    
    print(f"Dataset 1 shape: {df1.shape}")
    print(f"Dataset 2 shape: {df2.shape}")
    
    # Reshape from wide to long format
    def reshape_to_long(df, country_col, dataset_name):
        # Identify year columns (columns that contain year patterns)
        year_cols = [col for col in df.columns if any(year in col for year in ['199', '200', '201', '202'])]
        
        # Melt to long format
        df_long = df.melt(
            id_vars=[country_col, 'Country Code'],
            value_vars=year_cols,
            var_name='Year',
            value_name=f'Value_{dataset_name}'
        )
        
        # Extract year from column names like "2011 [YR2011]" - FIXED escape sequence
        df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)
        
        # Rename country column
        df_long = df_long.rename(columns={country_col: "Country"})
        
        return df_long
    
    # Reshape both datasets
    df1_long = reshape_to_long(df1, country_col1, "ds1")
    df2_long = reshape_to_long(df2, country_col2, "ds2")
    
    print(f"After reshaping - df1_long shape: {df1_long.shape}")
    print(f"After reshaping - df2_long shape: {df2_long.shape}")
    
    # Merge the reshaped data
    how = "outer" if join_mode == "union" else "inner"
    
    merged = pd.merge(
        df1_long, df2_long,
        on=["Country", "Country Code", "Year"],
        how=how,
        suffixes=("", "_2")  # Remove suffix since we already added dataset names
    )
    
    # Drop duplicate columns from the merge
    cols_to_drop = [col for col in merged.columns if col.endswith('_2')]
    merged = merged.drop(columns=cols_to_drop)
    
    merged = merged.sort_values(["Country", "Year"]).reset_index(drop=True)
    
    print(f"Final merged shape: {merged.shape}")
    print(f"Final columns: {list(merged.columns)}")
    
    if output_path:
        merged.to_csv(output_path, index=False, encoding="utf-8")
        print("Saved:", output_path)

    return merged


# RUN IT again with the fix
merged_union = merge_two_panels(
    "Datasets/Education Expenditure/7a5010d8-8a7d-4f21-b71a-fee74d8618e6_Series - Metadata.csv",
    "Datasets/Education Expenditure/Expenditure on Education as %age of GDP - 7a5010d8-8a7d-4f21-b71a-fee74d8618e6_Data.csv",
    country_col1="Country Name",
    country_col2="Country Name", 
    join_mode="union",
    output_path="Education Expenditure.csv"
)

# Display sample of the result
print("\nSample of merged data:")
print(merged_union.head(10))

Dataset 1 shape: (273, 18)
Dataset 2 shape: (266, 16)
After reshaping - df1_long shape: (3822, 4)
After reshaping - df2_long shape: (3724, 4)
Final merged shape: (3822, 5)
Final columns: ['Country', 'Country Code', 'Year', 'Value_ds1', 'Value_ds2']
Saved: Education Expenditure.csv

Sample of merged data:
       Country Country Code  Year         Value_ds1    Value_ds2
0  Afghanistan          AFG  2011   3.4620099067688  3.462009907
1  Afghanistan          AFG  2012  2.60419988632202  2.604199886
2  Afghanistan          AFG  2013  3.45445990562439  3.454459906
3  Afghanistan          AFG  2014  3.69521999359131  3.695219994
4  Afghanistan          AFG  2015   3.2558000087738  3.255800009
5  Afghanistan          AFG  2016  4.54397010803223  4.543970108
6  Afghanistan          AFG  2017  4.34319019317627  4.343190193
7  Afghanistan          AFG  2018                ..           ..
8  Afghanistan          AFG  2019                ..           ..
9  Afghanistan          AFG  2020           

### Combine Fertility Rate

In [22]:
import pandas as pd

def merge_fertility_datasets(
    metadata_path,
    data_path, 
    output_path=None
):
    """
    Merge two fertility rate datasets from World Bank format
    
    Parameters:
    metadata_path: path to the Series - Metadata.csv file
    data_path: path to the Fertility Rate.csv file  
    output_path: optional path to save merged dataset
    """
    
    # Read both datasets with correct encoding
    df_meta = pd.read_csv(metadata_path, encoding="cp1252")
    df_data = pd.read_csv(data_path, encoding="cp1252")
    
    print(f"Metadata dataset shape: {df_meta.shape}")
    print(f"Data dataset shape: {df_data.shape}")
    
    # Reshape both datasets from wide to long format
    def reshape_to_long(df, dataset_name):
        # Identify year columns (columns that contain year patterns)
        year_cols = [col for col in df.columns if any(year in col for year in ['199', '200', '201', '202'])]
        
        # Melt to long format
        df_long = df.melt(
            id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
            value_vars=year_cols,
            var_name='Year',
            value_name=f'Fertility_Rate_{dataset_name}'
        )
        
        # Extract year from column names like "2011 [YR2011]"
        df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)
        
        return df_long
    
    # Reshape both datasets
    df_meta_long = reshape_to_long(df_meta, "meta")
    df_data_long = reshape_to_long(df_data, "data")
    
    print(f"After reshaping - Metadata shape: {df_meta_long.shape}")
    print(f"After reshaping - Data shape: {df_data_long.shape}")
    
    # Merge the datasets on common columns
    merged = pd.merge(
        df_meta_long,
        df_data_long,
        on=['Country Name', 'Country Code', 'Series Name', 'Series Code', 'Year'],
        how='outer',
        suffixes=('', '_drop')
    )
    
    # Drop any duplicate columns from the merge
    cols_to_drop = [col for col in merged.columns if col.endswith('_drop')]
    merged = merged.drop(columns=cols_to_drop)
    
    # Sort and clean up
    merged = merged.sort_values(['Country Name', 'Year']).reset_index(drop=True)
    
    print(f"Final merged shape: {merged.shape}")
    print(f"Final columns: {list(merged.columns)}")
    
    if output_path:
        merged.to_csv(output_path, index=False, encoding="utf-8")
        print(f"Saved merged dataset to: {output_path}")
    
    return merged

# Merge the fertility rate datasets
fertility_merged = merge_fertility_datasets(
    metadata_path="Datasets/Fertility Rate/4735b18c-8ee7-43b7-973d-3c7c67940661_Series - Metadata.csv",
    data_path="Datasets/Fertility Rate/Fertility Rate - Fertility Rate.csv",
    output_path="Fertility_Rate_Merged.csv"
)

# Display sample of the result
print("\nSample of merged fertility data:")
print(fertility_merged.head(10))

Metadata dataset shape: (273, 18)
Data dataset shape: (222, 18)
After reshaping - Metadata shape: (3822, 6)
After reshaping - Data shape: (3108, 6)
Final merged shape: (3906, 7)
Final columns: ['Country Name', 'Country Code', 'Series Name', 'Series Code', 'Year', 'Fertility_Rate_meta', 'Fertility_Rate_data']
Saved merged dataset to: Fertility_Rate_Merged.csv

Sample of merged fertility data:
  Country Name Country Code                               Series Name  \
0  Afghanistan          AFG  Fertility rate, total (births per woman)   
1  Afghanistan          AFG  Fertility rate, total (births per woman)   
2  Afghanistan          AFG  Fertility rate, total (births per woman)   
3  Afghanistan          AFG  Fertility rate, total (births per woman)   
4  Afghanistan          AFG  Fertility rate, total (births per woman)   
5  Afghanistan          AFG  Fertility rate, total (births per woman)   
6  Afghanistan          AFG  Fertility rate, total (births per woman)   
7  Afghanistan       

### Combine Final Consumption Per Capita

In [23]:
import pandas as pd

def merge_consumption_datasets(
    metadata_path,
    data_path, 
    output_path=None
):
    """
    Merge two final consumption expenditure per capita datasets from World Bank format
    
    Parameters:
    metadata_path: path to the Series - Metadata.csv file
    data_path: path to the Final consumption expenditure per capita.csv file  
    output_path: optional path to save merged dataset
    """
    
    # Read both datasets with correct encoding
    df_meta = pd.read_csv(metadata_path, encoding="cp1252")
    df_data = pd.read_csv(data_path, encoding="cp1252")
    
    print(f"Metadata dataset shape: {df_meta.shape}")
    print(f"Data dataset shape: {df_data.shape}")
    
    # Reshape both datasets from wide to long format
    def reshape_to_long(df, dataset_name):
        # Identify year columns (columns that contain year patterns)
        year_cols = [col for col in df.columns if any(year in col for year in ['199', '200', '201', '202'])]
        
        # Melt to long format
        if 'Series Name' in df.columns and 'Series Code' in df.columns:
            # For metadata file with series info
            df_long = df.melt(
                id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
                value_vars=year_cols,
                var_name='Year',
                value_name=f'Consumption_Per_Capita_{dataset_name}'
            )
        else:
            # For data file without series info
            df_long = df.melt(
                id_vars=['Country Name', 'Country Code'],
                value_vars=year_cols,
                var_name='Year',
                value_name=f'Consumption_Per_Capita_{dataset_name}'
            )
        
        # Extract year from column names like "2011 [YR2011]"
        df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)
        
        return df_long
    
    # Reshape both datasets
    df_meta_long = reshape_to_long(df_meta, "meta")
    df_data_long = reshape_to_long(df_data, "data")
    
    print(f"After reshaping - Metadata shape: {df_meta_long.shape}")
    print(f"After reshaping - Data shape: {df_data_long.shape}")
    
    # Merge the datasets on common columns
    merged = pd.merge(
        df_meta_long,
        df_data_long,
        on=['Country Name', 'Country Code', 'Year'],
        how='outer',
        suffixes=('', '_drop')
    )
    
    # Drop any duplicate columns from the merge
    cols_to_drop = [col for col in merged.columns if col.endswith('_drop')]
    merged = merged.drop(columns=cols_to_drop)
    
    # Sort and clean up
    merged = merged.sort_values(['Country Name', 'Year']).reset_index(drop=True)
    
    print(f"Final merged shape: {merged.shape}")
    print(f"Final columns: {list(merged.columns)}")
    
    if output_path:
        merged.to_csv(output_path, index=False, encoding="utf-8")
        print(f"Saved merged dataset to: {output_path}")
    
    return merged

# Merge the consumption per capita datasets
consumption_merged = merge_consumption_datasets(
    metadata_path="Datasets/Final Consumption Per Capita/615ed1f2-f56d-4f14-8784-f80fb40ed40e_Series - Metadata.csv",
    data_path="Datasets/Final Consumption Per Capita/Final consumption expenditure per capita (constant 2015 US$) - 615ed1f2-f56d-4f14-8784-f80fb40ed40e_Data.csv",
    output_path="Final_Consumption_Per_Capita_Merged.csv"
)

# Display sample of the result
print("\nSample of merged consumption data:")
print(consumption_merged.head(10))

Metadata dataset shape: (273, 18)
Data dataset shape: (217, 16)
After reshaping - Metadata shape: (3822, 6)
After reshaping - Data shape: (3038, 4)
Final merged shape: (3822, 7)
Final columns: ['Country Name', 'Country Code', 'Series Name', 'Series Code', 'Year', 'Consumption_Per_Capita_meta', 'Consumption_Per_Capita_data']
Saved merged dataset to: Final_Consumption_Per_Capita_Merged.csv

Sample of merged consumption data:
  Country Name Country Code  \
0  Afghanistan          AFG   
1  Afghanistan          AFG   
2  Afghanistan          AFG   
3  Afghanistan          AFG   
4  Afghanistan          AFG   
5  Afghanistan          AFG   
6  Afghanistan          AFG   
7  Afghanistan          AFG   
8  Afghanistan          AFG   
9  Afghanistan          AFG   

                                         Series Name        Series Code  Year  \
0  Households and NPISHs Final consumption expend...  NE.CON.PRVT.PC.KD  2011   
1  Households and NPISHs Final consumption expend...  NE.CON.PRVT.PC.

### Extract GDP per Capita 

In [24]:
import pandas as pd

def extract_complete_panel(
    data_path,
    output_path=None
):
    """
    Extract complete panel data with GDP per capita and happiness scores
    
    Parameters:
    data_path: path to the gdp-vs-happiness.csv file
    output_path: optional path to save cleaned dataset
    """
    
    # Read the dataset
    df = pd.read_csv(data_path)
    
    print(f"Original dataset shape: {df.shape}")
    
    # Filter for country-level data (remove regional aggregates)
    df_countries = df[df['Code'].notna()].copy()
    
    print(f"After filtering countries: {df_countries.shape}")
    
    # Select relevant columns
    panel_data = df_countries[[
        'Entity', 'Code', 'Year', 
        'Cantril ladder score', 
        'GDP per capita, PPP (constant 2021 international $)',
        'World regions according to OWID'
    ]].copy()
    
    # Rename columns for consistency
    panel_data = panel_data.rename(columns={
        'Entity': 'Country',
        'Code': 'Country_Code',
        'Cantril ladder score': 'Life_Satisfaction',
        'GDP per capita, PPP (constant 2021 international $)': 'GDP_per_capita_PPP',
        'World regions according to OWID': 'Region'
    })
    
    # Convert to numeric
    panel_data['GDP_per_capita_PPP'] = pd.to_numeric(panel_data['GDP_per_capita_PPP'], errors='coerce')
    panel_data['Life_Satisfaction'] = pd.to_numeric(panel_data['Life_Satisfaction'], errors='coerce')
    
    # Sort by country and year
    panel_data = panel_data.sort_values(['Country', 'Year']).reset_index(drop=True)
    
    print(f"Final panel dataset shape: {panel_data.shape}")
    print(f"Year range: {panel_data['Year'].min()} - {panel_data['Year'].max()}")
    print(f"Number of countries: {panel_data['Country'].nunique()}")
    
    # Show data availability
    print(f"\nData availability:")
    print(f"Rows with GDP data: {panel_data['GDP_per_capita_PPP'].notna().sum()}")
    print(f"Rows with happiness data: {panel_data['Life_Satisfaction'].notna().sum()}")
    print(f"Rows with both: {panel_data[panel_data['GDP_per_capita_PPP'].notna() & panel_data['Life_Satisfaction'].notna()].shape[0]}")
    
    if output_path:
        panel_data.to_csv(output_path, index=False, encoding="utf-8")
        print(f"\nSaved complete panel data to: {output_path}")
    
    return panel_data

# Extract complete panel data
complete_panel = extract_complete_panel(
    data_path="Datasets/Get GDP per Capita from this dataset/gdp-vs-happiness.csv",
    output_path="GDP_Happiness_Panel.csv"
)

# Display sample
print("\nSample of complete panel data:")
print(complete_panel.head(15))

Original dataset shape: (7448, 6)
After filtering countries: (6935, 6)
Final panel dataset shape: (6935, 6)
Year range: 1990 - 2024
Number of countries: 260

Data availability:
Rows with GDP data: 6816
Rows with happiness data: 1982
Rows with both: 1920

Saved complete panel data to: GDP_Happiness_Panel.csv

Sample of complete panel data:
        Country Country_Code  Year  Life_Satisfaction  GDP_per_capita_PPP  \
0   Afghanistan          AFG  2000                NaN           1617.8264   
1   Afghanistan          AFG  2001                NaN           1454.1108   
2   Afghanistan          AFG  2002                NaN           1774.3087   
3   Afghanistan          AFG  2003                NaN           1815.9282   
4   Afghanistan          AFG  2004                NaN           1776.9182   
5   Afghanistan          AFG  2005                NaN           1908.1147   
6   Afghanistan          AFG  2006                NaN           1929.7239   
7   Afghanistan          AFG  2007         

### Combine Health Expenditure per Capita

In [25]:
import pandas as pd

def merge_health_expenditure_datasets(
    metadata_path,
    data_path, 
    output_path=None
):
    """
    Merge two health expenditure per capita datasets from World Bank format
    
    Parameters:
    metadata_path: path to the Health Expenditure per Capita - Metadata.csv file
    data_path: path to the Health Expenditure per Capita.csv file  
    output_path: optional path to save merged dataset
    """
    
    # Read both datasets with correct encoding
    df_meta = pd.read_csv(metadata_path, encoding="cp1252")
    df_data = pd.read_csv(data_path, encoding="cp1252")
    
    print(f"Metadata dataset shape: {df_meta.shape}")
    print(f"Data dataset shape: {df_data.shape}")
    
    # Reshape both datasets from wide to long format
    def reshape_to_long(df, dataset_name):
        # Identify year columns (columns that contain year patterns)
        year_cols = [col for col in df.columns if any(year in col for year in ['199', '200', '201', '202'])]
        
        # Melt to long format
        if 'Series Name' in df.columns and 'Series Code' in df.columns:
            # For metadata file with series info
            df_long = df.melt(
                id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
                value_vars=year_cols,
                var_name='Year',
                value_name=f'Health_Expenditure_Per_Capita_{dataset_name}'
            )
        else:
            # For data file without series info
            df_long = df.melt(
                id_vars=['Country Name', 'Country Code'],
                value_vars=year_cols,
                var_name='Year',
                value_name=f'Health_Expenditure_Per_Capita_{dataset_name}'
            )
        
        # Extract year from column names like "2011 [YR2011]"
        df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)
        
        return df_long
    
    # Reshape both datasets
    df_meta_long = reshape_to_long(df_meta, "meta")
    df_data_long = reshape_to_long(df_data, "data")
    
    print(f"After reshaping - Metadata shape: {df_meta_long.shape}")
    print(f"After reshaping - Data shape: {df_data_long.shape}")
    
    # Merge the datasets on common columns
    merged = pd.merge(
        df_meta_long,
        df_data_long,
        on=['Country Name', 'Country Code', 'Year'],
        how='outer',
        suffixes=('', '_drop')
    )
    
    # Drop any duplicate columns from the merge
    cols_to_drop = [col for col in merged.columns if col.endswith('_drop')]
    merged = merged.drop(columns=cols_to_drop)
    
    # Sort and clean up
    merged = merged.sort_values(['Country Name', 'Year']).reset_index(drop=True)
    
    print(f"Final merged shape: {merged.shape}")
    print(f"Final columns: {list(merged.columns)}")
    
    # Display data summary
    print(f"\nData summary:")
    print(f"Year range: {merged['Year'].min()} - {merged['Year'].max()}")
    print(f"Number of countries: {merged['Country Name'].nunique()}")
    print(f"Total observations: {len(merged)}")
    print(f"Observations with health expenditure data: {merged['Health_Expenditure_Per_Capita_meta'].notna().sum()}")
    
    if output_path:
        merged.to_csv(output_path, index=False, encoding="utf-8")
        print(f"\nSaved merged health expenditure data to: {output_path}")
    
    return merged

# Merge the health expenditure per capita datasets
health_expenditure_merged = merge_health_expenditure_datasets(
    metadata_path="Datasets/Health Expenditure per Capita/Health Expenditure per Capita - Metadata.csv",
    data_path="Datasets/Health Expenditure per Capita/Health Expenditure per Capita.csv",
    output_path="Health_Expenditure_Per_Capita_Merged.csv"
)

# Display sample of the result
print("\nSample of merged health expenditure data:")
print(health_expenditure_merged.head(10))

# Optional: Show summary by year
print(f"\nObservations by year:")
year_counts = health_expenditure_merged.groupby('Year').size()
print(year_counts)

Metadata dataset shape: (273, 18)
Data dataset shape: (217, 16)
After reshaping - Metadata shape: (3822, 6)
After reshaping - Data shape: (3038, 4)
Final merged shape: (3822, 7)
Final columns: ['Country Name', 'Country Code', 'Series Name', 'Series Code', 'Year', 'Health_Expenditure_Per_Capita_meta', 'Health_Expenditure_Per_Capita_data']

Data summary:
Year range: 2011 - 2024
Number of countries: 268
Total observations: 3822
Observations with health expenditure data: 3746

Saved merged health expenditure data to: Health_Expenditure_Per_Capita_Merged.csv

Sample of merged health expenditure data:
  Country Name Country Code  \
0  Afghanistan          AFG   
1  Afghanistan          AFG   
2  Afghanistan          AFG   
3  Afghanistan          AFG   
4  Afghanistan          AFG   
5  Afghanistan          AFG   
6  Afghanistan          AFG   
7  Afghanistan          AFG   
8  Afghanistan          AFG   
9  Afghanistan          AFG   

                                         Series Name   

### Combine Homicides

In [28]:
import pandas as pd

def merge_homicide_datasets(
    metadata_path,
    data_path, 
    output_path=None
):
    """
    Merge two intentional homicide rate datasets from World Bank format
    
    Parameters:
    metadata_path: path to the Series - Metadata.csv file
    data_path: path to the Intentional Homicides per 100k.csv file  
    output_path: optional path to save merged dataset
    """
    
    # Read both datasets with error handling
    try:
        df_meta = pd.read_csv(metadata_path, encoding="cp1252", on_bad_lines='warn')
    except:
        df_meta = pd.read_csv(metadata_path, encoding="cp1252", sep=None, engine='python')
    
    try:
        df_data = pd.read_csv(data_path, encoding="cp1252", on_bad_lines='warn')
    except:
        df_data = pd.read_csv(data_path, encoding="cp1252", sep=None, engine='python')
    
    print(f"Metadata dataset shape: {df_meta.shape}")
    print(f"Data dataset shape: {df_data.shape}")
    
    # Reshape both datasets from wide to long format
    def reshape_to_long(df, dataset_name):
        # Identify year columns (columns that contain year patterns)
        year_cols = [col for col in df.columns if any(year in col for year in ['199', '200', '201', '202'])]
        
        # Melt to long format
        if 'Series Name' in df.columns and 'Series Code' in df.columns:
            # For metadata file with series info
            df_long = df.melt(
                id_vars=['Country Name', 'Country Code', 'Series Name', 'Series Code'],
                value_vars=year_cols,
                var_name='Year',
                value_name=f'Homicide_Rate_{dataset_name}'
            )
        else:
            # For data file without series info
            df_long = df.melt(
                id_vars=['Country Name', 'Country Code'],
                value_vars=year_cols,
                var_name='Year',
                value_name=f'Homicide_Rate_{dataset_name}'
            )
        
        # Extract year from column names like "2011 [YR2011]"
        df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)
        
        # Convert homicide rate to numeric, handling ".." as missing values
        df_long[f'Homicide_Rate_{dataset_name}'] = pd.to_numeric(
            df_long[f'Homicide_Rate_{dataset_name}'], 
            errors='coerce'
        )
        
        return df_long
    
    # Reshape both datasets
    df_meta_long = reshape_to_long(df_meta, "meta")
    df_data_long = reshape_to_long(df_data, "data")
    
    print(f"After reshaping - Metadata shape: {df_meta_long.shape}")
    print(f"After reshaping - Data shape: {df_data_long.shape}")
    
    # Merge the datasets on common columns
    merged = pd.merge(
        df_meta_long,
        df_data_long,
        on=['Country Name', 'Country Code', 'Year'],
        how='outer',
        suffixes=('', '_drop')
    )
    
    # Drop any duplicate columns from the merge
    cols_to_drop = [col for col in merged.columns if col.endswith('_drop')]
    merged = merged.drop(columns=cols_to_drop)
    
    # Sort and clean up
    merged = merged.sort_values(['Country Name', 'Year']).reset_index(drop=True)
    
    print(f"Final merged shape: {merged.shape}")
    print(f"Final columns: {list(merged.columns)}")
    
    # Display data summary
    print(f"\nData summary:")
    print(f"Year range: {merged['Year'].min()} - {merged['Year'].max()}")
    print(f"Number of countries: {merged['Country Name'].nunique()}")
    print(f"Total observations: {len(merged)}")
    print(f"Observations with homicide rate data: {merged['Homicide_Rate_meta'].notna().sum()}")
    
    # Show some statistics about homicide rates
    if merged['Homicide_Rate_meta'].notna().sum() > 0:
        print(f"\nHomicide rate statistics (per 100,000 people):")
        print(f"Mean: {merged['Homicide_Rate_meta'].mean():.2f}")
        print(f"Median: {merged['Homicide_Rate_meta'].median():.2f}")
        print(f"Min: {merged['Homicide_Rate_meta'].min():.2f}")
        print(f"Max: {merged['Homicide_Rate_meta'].max():.2f}")
    
    if output_path:
        merged.to_csv(output_path, index=False, encoding="utf-8")
        print(f"\nSaved merged homicide rate data to: {output_path}")
    
    return merged

# Merge the homicide rate datasets
homicide_merged = merge_homicide_datasets(
    metadata_path="Datasets/Homicides/2715e2f7-b0fd-4e81-a6dd-30f3b5bd2882_Series - Metadata.csv",
    data_path="Datasets/Homicides/Intentional Homicides per 100k - Intentional Homicides per 100k.csv",
    output_path="Homicide_Rate_Merged.csv"
)

# Display sample of the result
print("\nSample of merged homicide rate data:")
print(homicide_merged.head(10))

# Optional: Show countries with highest average homicide rates
print(f"\nCountries with highest average homicide rates:")
if homicide_merged['Homicide_Rate_meta'].notna().sum() > 0:
    avg_homicide = homicide_merged.groupby('Country Name')['Homicide_Rate_meta'].mean().sort_values(ascending=False)
    print(avg_homicide.head(10))

Metadata dataset shape: (272, 18)
Data dataset shape: (266, 16)
After reshaping - Metadata shape: (3808, 6)
After reshaping - Data shape: (3724, 4)
Final merged shape: (3808, 7)
Final columns: ['Country Name', 'Country Code', 'Series Name', 'Series Code', 'Year', 'Homicide_Rate_meta', 'Homicide_Rate_data']

Data summary:
Year range: 2011 - 2024
Number of countries: 267
Total observations: 3808
Observations with homicide rate data: 1967

Homicide rate statistics (per 100,000 people):
Mean: 7.78
Median: 3.48
Min: 0.07
Max: 107.64

Saved merged homicide rate data to: Homicide_Rate_Merged.csv

Sample of merged homicide rate data:
  Country Name Country Code                                 Series Name  \
0  Afghanistan          AFG  Intentional homicides (per 100,000 people)   
1  Afghanistan          AFG  Intentional homicides (per 100,000 people)   
2  Afghanistan          AFG  Intentional homicides (per 100,000 people)   
3  Afghanistan          AFG  Intentional homicides (per 100,000 pe

C:\Users\ibrahim\AppData\Local\Temp\ipykernel_8652\4051940593.py:19: ParserWarning: Skipping line 274: expected 18 fields, saw 19

  df_meta = pd.read_csv(metadata_path, encoding="cp1252", on_bad_lines='warn')


### Extract Inflation of Consumer Prices

In [30]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv('Datasets/inflation-of-consumer-prices/inflation-of-consumer-prices.csv')

# 1. Clean column names and create a clean copy to avoid SettingWithCopyWarning
df_clean = df.copy()
df_clean.columns = ['Entity', 'Code', 'Year', 'Inflation_Consumer_Prices_Annual_Pct']

# 2. Remove any rows with missing critical values
df_clean = df_clean.dropna(subset=['Entity', 'Code', 'Year', 'Inflation_Consumer_Prices_Annual_Pct'])

# 3. Add decade column
df_clean['Decade'] = (df_clean['Year'] // 10) * 10

# 4. Categorize inflation levels
def categorize_inflation(value):
    if value < 0:
        return 'Deflation'
    elif value < 2:
        return 'Low'
    elif value < 5:
        return 'Moderate'
    elif value < 10:
        return 'High'
    else:
        return 'Hyperinflation'

df_clean['Inflation_Category'] = df_clean['Inflation_Consumer_Prices_Annual_Pct'].apply(categorize_inflation)

# 5. Calculate rolling statistics for each country
df_clean['Inflation_3yr_Avg'] = df_clean.groupby('Entity')['Inflation_Consumer_Prices_Annual_Pct'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# 6. Add year-over-year change flag
df_clean['YoY_Increase'] = df_clean.groupby('Entity')['Inflation_Consumer_Prices_Annual_Pct'].transform(
    lambda x: x > x.shift(1)
)

# 7. Sort the data logically
df_clean = df_clean.sort_values(['Entity', 'Year']).reset_index(drop=True)

# Display improved dataset info
print("Improved Dataset Info:")
print(f"Shape: {df_clean.shape}")
print(f"Countries: {df_clean['Entity'].nunique()}")
print(f"Years: {df_clean['Year'].min()} - {df_clean['Year'].max()}")
print("\nFirst 10 rows:")
print(df_clean.head(10))

# Save improved dataset
df_clean.to_csv('improved_inflation_data.csv', index=False)
print("\nImproved dataset saved as 'improved_inflation_data.csv'")

Improved Dataset Info:
Shape: (8980, 8)
Countries: 193
Years: 1960 - 2024

First 10 rows:
        Entity Code  Year  Inflation_Consumer_Prices_Annual_Pct  Decade  \
0  Afghanistan  AFG  2005                             12.686269    2000   
1  Afghanistan  AFG  2006                              6.784596    2000   
2  Afghanistan  AFG  2007                              8.680571    2000   
3  Afghanistan  AFG  2008                             26.418665    2000   
4  Afghanistan  AFG  2009                             -6.811161    2000   
5  Afghanistan  AFG  2010                              2.178538    2010   
6  Afghanistan  AFG  2011                             11.804186    2010   
7  Afghanistan  AFG  2012                              6.441213    2010   
8  Afghanistan  AFG  2013                              7.385772    2010   
9  Afghanistan  AFG  2014                              4.673996    2010   

  Inflation_Category  Inflation_3yr_Avg  YoY_Increase  
0     Hyperinflation        

### Combine Internet Users Data

In [32]:
import pandas as pd
import numpy as np

# Read the main Internet Users dataset
internet_df = pd.read_csv('Datasets/Internet Users/Internet Users as Percentage of Total Population.csv', skiprows=4)

# Read the country metadata - check actual column names
country_meta_df = pd.read_csv('Datasets/Internet Users/Metadata_Country_API_IT.NET.USER.ZS_DS2_en_csv_v2_129784.csv')

# Read the indicator metadata
indicator_meta_df = pd.read_csv('Datasets/Internet Users/Metadata_Indicator_API_IT.NET.USER.ZS_DS2_en_csv_v2_129784.csv')

# Display column names to debug
print("Internet DataFrame columns:", internet_df.columns.tolist())
print("Country Metadata columns:", country_meta_df.columns.tolist())
print("Indicator Metadata columns:", indicator_meta_df.columns.tolist())

# Clean the main internet dataset
internet_df = internet_df.dropna(how='all', axis=1)
internet_df = internet_df.dropna(how='all', axis=0)

# Rename columns for consistency based on actual column names
internet_df = internet_df.rename(columns={
    'Country Name': 'Country', 
    'Country Code': 'Code'
})

# Reshape from wide to long format
year_columns = [str(year) for year in range(2011, 2025)]
internet_long = pd.melt(
    internet_df, 
    id_vars=['Country', 'Code'], 
    value_vars=year_columns,
    var_name='Year', 
    value_name='Internet_Users_Pct'
)

# Convert Year to integer and handle missing values
internet_long['Year'] = internet_long['Year'].astype(int)
internet_long['Internet_Users_Pct'] = pd.to_numeric(internet_long['Internet_Users_Pct'], errors='coerce')

# Clean country metadata column names (remove quotes and spaces)
country_meta_df.columns = country_meta_df.columns.str.replace('"', '').str.strip()
print("Cleaned Country Metadata columns:", country_meta_df.columns.tolist())

# Rename country metadata columns for merge
country_meta_df = country_meta_df.rename(columns={
    'Country Code': 'Code',
    'Region': 'Region',
    'IncomeGroup': 'Income_Group',
    'SpecialNotes': 'Special_Notes',
    'TableName': 'Table_Name'
})

# Merge with country metadata
merged_df = pd.merge(
    internet_long,
    country_meta_df,
    on='Code',
    how='left'
)

# Clean indicator metadata column names
indicator_meta_df.columns = indicator_meta_df.columns.str.replace('"', '').str.strip()

# Add indicator metadata information
merged_df['Indicator_Code'] = 'IT.NET.USER.ZS'
merged_df['Indicator_Name'] = 'Individuals using the Internet (% of population)'
merged_df['Source_Organization'] = 'International Telecommunication Union (ITU)'

# Sort and clean the final dataset
merged_df = merged_df.sort_values(['Country', 'Year']).reset_index(drop=True)

# Select and order columns for final output
final_columns = [
    'Country', 'Code', 'Year', 'Internet_Users_Pct', 
    'Region', 'Income_Group', 'Special_Notes',
    'Indicator_Code', 'Indicator_Name', 'Source_Organization'
]

final_df = merged_df[final_columns]

# Display results
print("\nMerged Dataset Info:")
print(f"Shape: {final_df.shape}")
print(f"Countries: {final_df['Country'].nunique()}")
print(f"Years: {final_df['Year'].min()} - {final_df['Year'].max()}")
print(f"Total records: {len(final_df):,}")

print("\nFirst 15 rows of merged data:")
print(final_df.head(15))

print("\nMissing values summary:")
print(final_df.isnull().sum())

# Save the merged dataset
final_df.to_csv('merged_internet_users_data.csv', index=False)
print("\nMerged dataset saved as 'merged_internet_users_data.csv'")

Internet DataFrame columns: ['Country Name', 'Country Code', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Country Metadata columns: ['Country Code', 'Region', 'IncomeGroup', 'SpecialNotes', 'TableName', 'Unnamed: 5']
Indicator Metadata columns: ['INDICATOR_CODE', 'INDICATOR_NAME', 'SOURCE_NOTE', 'SOURCE_ORGANIZATION', 'Unnamed: 4']
Cleaned Country Metadata columns: ['Country Code', 'Region', 'IncomeGroup', 'SpecialNotes', 'TableName', 'Unnamed: 5']

Merged Dataset Info:
Shape: (3724, 10)
Countries: 266
Years: 2011 - 2024
Total records: 3,724

First 15 rows of merged data:
                        Country Code  Year  Internet_Users_Pct  \
0                   Afghanistan  AFG  2011             5.00000   
1                   Afghanistan  AFG  2012             5.45455   
2                   Afghanistan  AFG  2013             5.90000   
3                   Afghanistan  AFG  2014             7.00000   
4                   Afgh

### Extract Life Expectancy at Birth

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Read the dataset
df = pd.read_csv('Datasets/Life Expectancy at Birth/life-expectancy.csv')

# 1. Clean and enhance the dataset - KEEPING ALL DATA NUMERICAL
def enhance_life_expectancy_data(df):
    # Create a clean copy
    df_clean = df.copy()
    
    # Clean column names
    df_clean.columns = ['Entity', 'Code', 'Year', 'Life_Expectancy']
    
    # Remove rows with missing critical values
    df_clean = df_clean.dropna(subset=['Entity', 'Year', 'Life_Expectancy'])
    
    # Add decade for analysis (keeping it numerical)
    df_clean['Decade'] = (df_clean['Year'] // 10) * 10
    
    # Calculate life expectancy improvements (numerical calculations)
    df_clean = df_clean.sort_values(['Entity', 'Year'])
    
    # Year-over-year change
    df_clean['LE_Yearly_Change'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: x - x.shift(1)
    )
    
    # 5-year changes
    df_clean['LE_5yr_Change'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: x - x.shift(5)
    )
    
    # 10-year changes
    df_clean['LE_10yr_Change'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: x - x.shift(10)
    )
    
    # Calculate rolling averages (numerical)
    df_clean['LE_5yr_Avg'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )
    
    df_clean['LE_10yr_Avg'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: x.rolling(window=10, min_periods=1).mean()
    )
    
    # Calculate growth rates (percentage changes)
    df_clean['LE_Yearly_Growth_Rate'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: (x - x.shift(1)) / x.shift(1) * 100
    )
    
    # Flag significant changes (boolean flags based on numerical thresholds)
    df_clean['Major_Drop_5yr'] = df_clean['LE_5yr_Change'] < -3
    df_clean['Major_Increase_5yr'] = df_clean['LE_5yr_Change'] > 3
    df_clean['Stagnant_Growth'] = abs(df_clean['LE_5yr_Change']) < 1
    
    # Calculate volatility (standard deviation of last 5 years)
    df_clean['LE_5yr_Std'] = df_clean.groupby('Entity')['Life_Expectancy'].transform(
        lambda x: x.rolling(window=5, min_periods=3).std()
    )
    
    # Add century (numerical)
    df_clean['Century'] = (df_clean['Year'] // 100) + 1
    
    # Calculate distance from global average for each year
    global_avg_by_year = df_clean.groupby('Year')['Life_Expectancy'].transform('mean')
    df_clean['Deviation_From_Global_Avg'] = df_clean['Life_Expectancy'] - global_avg_by_year
    
    # Calculate percent rank within each year
    df_clean['Percentile_Rank_Year'] = df_clean.groupby('Year')['Life_Expectancy'].transform(
        lambda x: x.rank(pct=True) * 100
    )
    
    return df_clean.sort_values(['Entity', 'Year']).reset_index(drop=True)

# Apply enhancements
enhanced_df = enhance_life_expectancy_data(df)

# Display basic information
print("ENHANCED LIFE EXPECTANCY DATASET")
print("=" * 50)
print(f"Dataset Shape: {enhanced_df.shape}")
print(f"Time Period: {enhanced_df['Year'].min()} - {enhanced_df['Year'].max()}")
print(f"Number of Countries/Regions: {enhanced_df['Entity'].nunique()}")
print(f"Total Records: {len(enhanced_df):,}")

print("\nFirst 10 rows (showing numerical enhancements only):")
print(enhanced_df[['Entity', 'Year', 'Life_Expectancy', 'LE_Yearly_Change', 'LE_5yr_Change', 'LE_5yr_Avg', 'Deviation_From_Global_Avg']].head(10))

ENHANCED LIFE EXPECTANCY DATASET
Dataset Shape: (21565, 18)
Time Period: 1543 - 2023
Number of Countries/Regions: 265
Total Records: 21,565

First 10 rows (showing numerical enhancements only):
        Entity  Year  Life_Expectancy  LE_Yearly_Change  LE_5yr_Change  \
0  Afghanistan  1950          28.1563               NaN            NaN   
1  Afghanistan  1951          28.5836            0.4273            NaN   
2  Afghanistan  1952          29.0138            0.4302            NaN   
3  Afghanistan  1953          29.4521            0.4383            NaN   
4  Afghanistan  1954          29.6975            0.2454            NaN   
5  Afghanistan  1955          30.3660            0.6685         2.2097   
6  Afghanistan  1956          30.8303            0.4643         2.2467   
7  Afghanistan  1957          31.3451            0.5148         2.3313   
8  Afghanistan  1958          31.8400            0.4949         2.3879   
9  Afghanistan  1959          32.3365            0.4965         2.

### Extract Schooling Time

In [36]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv('Datasets/Mean Years of Schooling/Mean Years of Schooling (Education).csv', encoding='cp1252')

# 1. Clean and enhance the dataset with numerical features only
def enhance_education_data(df):
    # Create a clean copy
    df_clean = df.copy()
    
    # Clean column names for consistency
    df_clean.columns = ['Country_Code', 'Country', 'Year', 'Mean_Years_Schooling', 'Note']
    
    # Remove rows with missing critical values
    df_clean = df_clean.dropna(subset=['Country_Code', 'Country', 'Year', 'Mean_Years_Schooling'])
    
    # Sort by country and year for proper time series calculations
    df_clean = df_clean.sort_values(['Country', 'Year']).reset_index(drop=True)
    
    # Calculate educational progress metrics
    df_clean['Yearly_Change'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: x - x.shift(1)
    )
    
    df_clean['Yearly_Growth_Rate'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: ((x - x.shift(1)) / x.shift(1)) * 100
    )
    
    # Calculate multi-year changes
    df_clean['Change_3yr'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: x - x.shift(3)
    )
    
    df_clean['Change_5yr'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: x - x.shift(5)
    )
    
    # Calculate rolling averages for trend analysis
    df_clean['Rolling_Avg_3yr'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    
    df_clean['Rolling_Avg_5yr'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )
    
    # Calculate volatility (standard deviation of recent years)
    df_clean['Volatility_3yr'] = df_clean.groupby('Country')['Mean_Years_Schooling'].transform(
        lambda x: x.rolling(window=3, min_periods=2).std()
    )
    
    # Calculate acceleration (change in growth rate)
    df_clean['Growth_Acceleration'] = df_clean.groupby('Country')['Yearly_Growth_Rate'].transform(
        lambda x: x - x.shift(1)
    )
    
    # Calculate distance from global average for each year
    global_avg_by_year = df_clean.groupby('Year')['Mean_Years_Schooling'].transform('mean')
    df_clean['Deviation_From_Global_Avg'] = df_clean['Mean_Years_Schooling'] - global_avg_by_year
    
    # Calculate percentile rank within each year
    df_clean['Global_Percentile_Rank'] = df_clean.groupby('Year')['Mean_Years_Schooling'].transform(
        lambda x: x.rank(pct=True) * 100
    )
    
    # Calculate educational attainment gaps (distance from top performer each year)
    top_performer_by_year = df_clean.groupby('Year')['Mean_Years_Schooling'].transform('max')
    df_clean['Gap_From_Top_Performer'] = top_performer_by_year - df_clean['Mean_Years_Schooling']
    
    # Flag significant changes
    df_clean['Significant_Increase'] = df_clean['Yearly_Change'] > 0.5
    df_clean['Significant_Decrease'] = df_clean['Yearly_Change'] < -0.2
    df_clean['Stagnant_Growth'] = abs(df_clean['Yearly_Change']) < 0.05
    
    # Calculate years to reach educational milestones
    df_clean['Years_To_6_Years'] = np.where(df_clean['Mean_Years_Schooling'] < 6, 
                                           (6 - df_clean['Mean_Years_Schooling']) / df_clean['Yearly_Change'].clip(lower=0.01), 
                                           np.nan)
    
    df_clean['Years_To_12_Years'] = np.where(df_clean['Mean_Years_Schooling'] < 12, 
                                            (12 - df_clean['Mean_Years_Schooling']) / df_clean['Yearly_Change'].clip(lower=0.01), 
                                            np.nan)
    
    return df_clean

# Apply enhancements
enhanced_df = enhance_education_data(df)

# Display basic information
print("ENHANCED MEAN YEARS OF SCHOOLING DATASET")
print("=" * 60)
print(f"Dataset Shape: {enhanced_df.shape}")
print(f"Time Period: {enhanced_df['Year'].min()} - {enhanced_df['Year'].max()}")
print(f"Number of Countries: {enhanced_df['Country'].nunique()}")
print(f"Total Records: {len(enhanced_df):,}")
print(f"Global Average Schooling (Latest Year): {enhanced_df[enhanced_df['Year'] == enhanced_df['Year'].max()]['Mean_Years_Schooling'].mean():.2f} years")

print("\nFirst 10 rows (showing key numerical enhancements):")
display_cols = ['Country', 'Year', 'Mean_Years_Schooling', 'Yearly_Change', 'Yearly_Growth_Rate', 
                'Rolling_Avg_3yr', 'Global_Percentile_Rank', 'Gap_From_Top_Performer']
print(enhanced_df[display_cols].head(10))

ENHANCED MEAN YEARS OF SCHOOLING DATASET
Dataset Shape: (2498, 21)
Time Period: 2011 - 2023
Number of Countries: 193
Total Records: 2,498
Global Average Schooling (Latest Year): 9.17 years

First 10 rows (showing key numerical enhancements):
       Country  Year  Mean_Years_Schooling  Yearly_Change  Yearly_Growth_Rate  \
0  Afghanistan  2011              1.937043            NaN                 NaN   
1  Afghanistan  2012              1.983860       0.046817            2.416918   
2  Afghanistan  2013              2.030677       0.046817            2.359882   
3  Afghanistan  2014              2.077493       0.046817            2.305476   
4  Afghanistan  2015              2.124310       0.046817            2.253521   
5  Afghanistan  2016              2.268592       0.144282            6.791931   
6  Afghanistan  2017              2.412873       0.144282            6.359966   
7  Afghanistan  2018              2.557155       0.144282            5.979662   
8  Afghanistan  2019         

### Happiness Extraction

In [37]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv('Datasets/Main Dataset (Contains the Happiness Target Variable)/happiness-cantril-ladder.csv')

# Clean and enhance the dataset
def enhance_happiness_data(df):
    df_clean = df.copy()
    df_clean.columns = ['Entity', 'Code', 'Year', 'Happiness_Score']
    
    # Remove rows with missing critical values
    df_clean = df_clean.dropna(subset=['Entity', 'Year', 'Happiness_Score'])
    
    # Sort by entity and year
    df_clean = df_clean.sort_values(['Entity', 'Year']).reset_index(drop=True)
    
    # Calculate happiness changes
    df_clean['Yearly_Change'] = df_clean.groupby('Entity')['Happiness_Score'].transform(
        lambda x: x - x.shift(1)
    )
    
    df_clean['Yearly_Growth_Rate'] = df_clean.groupby('Entity')['Happiness_Score'].transform(
        lambda x: ((x - x.shift(1)) / x.shift(1)) * 100
    )
    
    # Calculate rolling averages
    df_clean['Rolling_Avg_3yr'] = df_clean.groupby('Entity')['Happiness_Score'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    
    # Global comparative metrics
    global_avg_by_year = df_clean.groupby('Year')['Happiness_Score'].transform('mean')
    df_clean['Deviation_From_Global_Avg'] = df_clean['Happiness_Score'] - global_avg_by_year
    
    df_clean['Global_Percentile_Rank'] = df_clean.groupby('Year')['Happiness_Score'].transform(
        lambda x: x.rank(pct=True) * 100
    )
    
    # Happiness category flags (boolean, not categorical)
    df_clean['Very_High_Happiness'] = df_clean['Happiness_Score'] > 7
    df_clean['Very_Low_Happiness'] = df_clean['Happiness_Score'] < 4
    
    return df_clean

# Apply enhancements
enhanced_df = enhance_happiness_data(df)

# Display results
print("ENHANCED HAPPINESS DATASET")
print("=" * 40)
print(f"Shape: {enhanced_df.shape}")
print(f"Years: {enhanced_df['Year'].min()}-{enhanced_df['Year'].max()}")
print(f"Countries/Regions: {enhanced_df['Entity'].nunique()}")
print(f"Latest Global Avg: {enhanced_df[enhanced_df['Year'] == enhanced_df['Year'].max()]['Happiness_Score'].mean():.2f}")

print("\nFirst 8 rows:")
print(enhanced_df[['Entity', 'Year', 'Happiness_Score', 'Yearly_Change', 'Global_Percentile_Rank']].head(8))

# Save enhanced dataset
enhanced_df.to_csv('enhanced_happiness_data.csv', index=False)
print("\nEnhanced dataset saved as 'enhanced_happiness_data.csv'")

ENHANCED HAPPINESS DATASET
Shape: (2112, 11)
Years: 2011-2024
Countries/Regions: 178
Latest Global Avg: 5.58

First 8 rows:
        Entity  Year  Happiness_Score  Yearly_Change  Global_Percentile_Rank
0  Afghanistan  2011           4.2580            NaN               16.167665
1  Afghanistan  2012           4.0400        -0.2180                8.383234
2  Afghanistan  2014           3.5750        -0.4650                3.550296
3  Afghanistan  2015           3.3600        -0.2150                2.380952
4  Afghanistan  2016           3.7940         0.4340                9.036145
5  Afghanistan  2017           3.6320        -0.1620                7.185629
6  Afghanistan  2018           3.2030        -0.4290                1.796407
7  Afghanistan  2019           2.5669        -0.6361                0.609756

Enhanced dataset saved as 'enhanced_happiness_data.csv'


### Extract Obesity

In [38]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv('Datasets/Obesity/share-of-adults-who-are-overweight.csv')

# Clean and enhance the dataset
def enhance_obesity_data(df):
    df_clean = df.copy()
    df_clean.columns = ['Entity', 'Code', 'Year', 'Overweight_Pct']
    
    # Remove rows with missing critical values
    df_clean = df_clean.dropna(subset=['Entity', 'Year', 'Overweight_Pct'])
    
    # Sort by entity and year
    df_clean = df_clean.sort_values(['Entity', 'Year']).reset_index(drop=True)
    
    # Calculate obesity changes and growth rates
    df_clean['Yearly_Change'] = df_clean.groupby('Entity')['Overweight_Pct'].transform(
        lambda x: x - x.shift(1)
    )
    
    df_clean['Yearly_Growth_Rate'] = df_clean.groupby('Entity')['Overweight_Pct'].transform(
        lambda x: ((x - x.shift(1)) / x.shift(1)) * 100
    )
    
    # Calculate decade changes
    df_clean['Decade_Change'] = df_clean.groupby('Entity')['Overweight_Pct'].transform(
        lambda x: x - x.shift(10)
    )
    
    # Calculate rolling averages
    df_clean['Rolling_Avg_5yr'] = df_clean.groupby('Entity')['Overweight_Pct'].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean()
    )
    
    # Global comparative metrics
    global_avg_by_year = df_clean.groupby('Year')['Overweight_Pct'].transform('mean')
    df_clean['Deviation_From_Global_Avg'] = df_clean['Overweight_Pct'] - global_avg_by_year
    
    df_clean['Global_Percentile_Rank'] = df_clean.groupby('Year')['Overweight_Pct'].transform(
        lambda x: x.rank(pct=True) * 100
    )
    
    # Obesity level flags (boolean)
    df_clean['High_Obesity'] = df_clean['Overweight_Pct'] > 50
    df_clean['Very_High_Obesity'] = df_clean['Overweight_Pct'] > 60
    df_clean['Critical_Level'] = df_clean['Overweight_Pct'] > 70
    
    # Calculate acceleration (change in growth rate)
    df_clean['Growth_Acceleration'] = df_clean.groupby('Entity')['Yearly_Growth_Rate'].transform(
        lambda x: x - x.shift(1)
    )
    
    return df_clean

# Apply enhancements
enhanced_df = enhance_obesity_data(df)

# Display results
print("ENHANCED OBESITY DATASET")
print("=" * 40)
print(f"Shape: {enhanced_df.shape}")
print(f"Years: {enhanced_df['Year'].min()}-{enhanced_df['Year'].max()}")
print(f"Countries/Regions: {enhanced_df['Entity'].nunique()}")
print(f"Latest Global Avg: {enhanced_df[enhanced_df['Year'] == enhanced_df['Year'].max()]['Overweight_Pct'].mean():.1f}%")

print("\nFirst 8 rows:")
print(enhanced_df[['Entity', 'Year', 'Overweight_Pct', 'Yearly_Change', 'Global_Percentile_Rank']].head(8))

# Save enhanced dataset
enhanced_df.to_csv('enhanced_obesity_data.csv', index=False)
print("\nEnhanced dataset saved as 'enhanced_obesity_data.csv'")

ENHANCED OBESITY DATASET
Shape: (6798, 14)
Years: 1990-2022
Countries/Regions: 206
Latest Global Avg: 52.4%

First 8 rows:
        Entity  Year  Overweight_Pct  Yearly_Change  Global_Percentile_Rank
0  Afghanistan  1990        10.13355            NaN               15.048544
1  Afghanistan  1991        10.73641        0.60286               15.533981
2  Afghanistan  1992        11.37092        0.63451               15.533981
3  Afghanistan  1993        12.03992        0.66900               16.019417
4  Afghanistan  1994        12.74325        0.70333               16.504854
5  Afghanistan  1995        13.48410        0.74085               17.475728
6  Afghanistan  1996        14.26634        0.78224               17.475728
7  Afghanistan  1997        15.09348        0.82714               17.475728

Enhanced dataset saved as 'enhanced_obesity_data.csv'


### Merge Population Data

In [43]:
import pandas as pd
import numpy as np

# Read both population datasets
df1 = pd.read_csv('Datasets/Population/5cdf4111-3d47-48f9-89ad-1bb7e3388037_Series - Metadata.csv')
df2 = pd.read_csv('Datasets/Population/Population - Population.csv')

# Clean and reshape the first dataset (with metadata)
df1_clean = df1.copy()
df1_clean = df1_clean[df1_clean['Series Name'] == 'Population, total']  # Keep only population data

# Reshape from wide to long format
year_cols = [col for col in df1_clean.columns if 'YR' in col]
df1_long = pd.melt(df1_clean, 
                  id_vars=['Country Name', 'Country Code'], 
                  value_vars=year_cols,
                  var_name='Year_Str', 
                  value_name='Population')

# Extract year from column names and convert to integer
df1_long['Year'] = df1_long['Year_Str'].str.extract(r'(\d+)').astype(int)
df1_long = df1_long.drop('Year_Str', axis=1)

# Convert Population to numeric, handling non-numeric values
df1_long['Population'] = pd.to_numeric(df1_long['Population'], errors='coerce')

# Clean the second dataset
df2_clean = df2.copy()

# Reshape second dataset from wide to long format
year_cols_df2 = [col for col in df2_clean.columns if 'YR' in col]
df2_long = pd.melt(df2_clean,
                  id_vars=['Country Name', 'Country Code'],
                  value_vars=year_cols_df2,
                  var_name='Year_Str',
                  value_name='Population')

# Extract year from column names
df2_long['Year'] = df2_long['Year_Str'].str.extract(r'(\d+)').astype(int)
df2_long = df2_long.drop('Year_Str', axis=1)

# Convert Population to numeric, handling non-numeric values
df2_long['Population'] = pd.to_numeric(df2_long['Population'], errors='coerce')

# Merge the datasets
merged_df = pd.concat([df1_long, df2_long], ignore_index=True)

# Remove duplicates (keeping first occurrence)
merged_df = merged_df.drop_duplicates(subset=['Country Name', 'Country Code', 'Year'])

# Remove rows with missing population values
merged_df = merged_df.dropna(subset=['Population'])

# Sort and clean
merged_df = merged_df.sort_values(['Country Name', 'Year']).reset_index(drop=True)
merged_df.columns = ['Country', 'Code', 'Population', 'Year']

# Add numerical enhancements
merged_df = merged_df.sort_values(['Country', 'Year'])
merged_df['Yearly_Change'] = merged_df.groupby('Country')['Population'].transform(
    lambda x: x - x.shift(1)
)

merged_df['Yearly_Growth_Rate'] = merged_df.groupby('Country')['Population'].transform(
    lambda x: ((x - x.shift(1)) / x.shift(1)) * 100
)

merged_df['Population_Millions'] = merged_df['Population'] / 1_000_000

# Display results
print("MERGED POPULATION DATASET")
print("=" * 40)
print(f"Shape: {merged_df.shape}")
print(f"Years: {merged_df['Year'].min()}-{merged_df['Year'].max()}")
print(f"Countries: {merged_df['Country'].nunique()}")
print(f"Total Records: {len(merged_df):,}")
print(f"Missing values removed: {len(df1_long) + len(df2_long) - len(merged_df)}")

print("\nFirst 10 rows:")
print(merged_df[['Country', 'Year', 'Population', 'Yearly_Change', 'Yearly_Growth_Rate']].head(10))

# Save merged dataset
merged_df.to_csv('merged_population_data.csv', index=False)
print("\nMerged dataset saved as 'merged_population_data.csv'")

MERGED POPULATION DATASET
Shape: (3710, 7)
Years: 2011-2024
Countries: 265
Total Records: 3,710
Missing values removed: 3052

First 10 rows:
       Country  Year  Population  Yearly_Change  Yearly_Growth_Rate
0  Afghanistan  2011  29347708.0            NaN                 NaN
1  Afghanistan  2012  30560034.0      1212326.0            4.130905
2  Afghanistan  2013  31622704.0      1062670.0            3.477319
3  Afghanistan  2014  32792523.0      1169819.0            3.699301
4  Afghanistan  2015  33831764.0      1039241.0            3.169140
5  Afghanistan  2016  34700612.0       868848.0            2.568143
6  Afghanistan  2017  35688935.0       988323.0            2.848143
7  Afghanistan  2018  36743039.0      1054104.0            2.953588
8  Afghanistan  2019  37856121.0      1113082.0            3.029368
9  Afghanistan  2020  39068979.0      1212858.0            3.203862

Merged dataset saved as 'merged_population_data.csv'


### Extract Poverty

In [44]:
import pandas as pd
import numpy as np

# Read the dataset
df = pd.read_csv('Datasets/Poverty/%age-of-population-living-in-extreme-poverty - OWID.csv')

# Clean and enhance the dataset
def enhance_poverty_data(df):
    df_clean = df.copy()
    df_clean.columns = ['Country', 'Year', 'Poverty_Rate']
    
    # Remove rows with missing critical values
    df_clean = df_clean.dropna(subset=['Country', 'Year', 'Poverty_Rate'])
    
    # Sort by country and year
    df_clean = df_clean.sort_values(['Country', 'Year']).reset_index(drop=True)
    
    # Calculate poverty changes
    df_clean['Yearly_Change'] = df_clean.groupby('Country')['Poverty_Rate'].transform(
        lambda x: x - x.shift(1)
    )
    
    df_clean['Yearly_Growth_Rate'] = df_clean.groupby('Country')['Poverty_Rate'].transform(
        lambda x: ((x - x.shift(1)) / x.shift(1).clip(lower=0.1)) * 100
    )
    
    # Calculate multi-year changes
    df_clean['Change_5yr'] = df_clean.groupby('Country')['Poverty_Rate'].transform(
        lambda x: x - x.shift(5)
    )
    
    df_clean['Change_10yr'] = df_clean.groupby('Country')['Poverty_Rate'].transform(
        lambda x: x - x.shift(10)
    )
    
    # Calculate rolling averages
    df_clean['Rolling_Avg_3yr'] = df_clean.groupby('Country')['Poverty_Rate'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    
    # Global comparative metrics
    global_avg_by_year = df_clean.groupby('Year')['Poverty_Rate'].transform('mean')
    df_clean['Deviation_From_Global_Avg'] = df_clean['Poverty_Rate'] - global_avg_by_year
    
    df_clean['Global_Percentile_Rank'] = df_clean.groupby('Year')['Poverty_Rate'].transform(
        lambda x: x.rank(pct=True) * 100
    )
    
    # Poverty level flags (boolean)
    df_clean['Extreme_Poverty'] = df_clean['Poverty_Rate'] > 10
    df_clean['Moderate_Poverty'] = (df_clean['Poverty_Rate'] > 5) & (df_clean['Poverty_Rate'] <= 10)
    df_clean['Low_Poverty'] = df_clean['Poverty_Rate'] <= 5
    df_clean['Zero_Poverty'] = df_clean['Poverty_Rate'] == 0
    
    # Calculate progress toward poverty reduction goals
    df_clean['Progress_To_Under_5_Pct'] = np.where(
        df_clean['Poverty_Rate'] > 5,
        ((5 - df_clean['Poverty_Rate']) / df_clean['Poverty_Rate']) * -100,
        np.where(df_clean['Poverty_Rate'] <= 5, 100, 0)
    )
    
    return df_clean

# Apply enhancements
enhanced_df = enhance_poverty_data(df)

# Display results
print("ENHANCED POVERTY DATASET")
print("=" * 40)
print(f"Shape: {enhanced_df.shape}")
print(f"Years: {enhanced_df['Year'].min()}-{enhanced_df['Year'].max()}")
print(f"Countries: {enhanced_df['Country'].nunique()}")
print(f"Total Records: {len(enhanced_df):,}")
print(f"Latest Global Avg: {enhanced_df[enhanced_df['Year'] == enhanced_df['Year'].max()]['Poverty_Rate'].mean():.1f}%")

print("\nFirst 10 rows:")
print(enhanced_df[['Country', 'Year', 'Poverty_Rate', 'Yearly_Change', 'Global_Percentile_Rank']].head(10))

# Save enhanced dataset
enhanced_df.to_csv('enhanced_poverty_data.csv', index=False)
print("\nEnhanced dataset saved as 'enhanced_poverty_data.csv'")

ENHANCED POVERTY DATASET
Shape: (2874, 15)
Years: 1963-2025
Countries: 195
Total Records: 2,874
Latest Global Avg: 15.7%

First 10 rows:
   Country  Year  Poverty_Rate  Yearly_Change  Global_Percentile_Rank
0  Albania  1996      2.967841            NaN               28.301887
1  Albania  2002      4.194722       1.226881               26.388889
2  Albania  2005      2.650218      -1.544504               39.560440
3  Albania  2008      0.827308      -1.822910               40.449438
4  Albania  2012      1.794590       0.967282               50.000000
5  Albania  2014      3.676140       1.881549               55.670103
6  Albania  2015      1.114864      -2.561275               43.434343
7  Albania  2016      1.140620       0.025756               43.298969
8  Albania  2017      1.444444       0.303824               53.763441
9  Albania  2018      1.010029      -0.434415               48.148148

Enhanced dataset saved as 'enhanced_poverty_data.csv'


### Merge Suicide Rate

In [46]:
import pandas as pd
import numpy as np

# Read both suicide rate datasets
df1 = pd.read_csv('Datasets/Suicide rates Per 100K/Part 1 - World_Population_Review (2022-23).csv')
df2 = pd.read_csv('Datasets/Suicide rates Per 100K/part 2 - (2011-2021)/death-rate-from-suicides-ghe (Only Need 2011-2021).csv')

# Clean and reshape the first dataset (2022-2023 data)
df1_clean = df1.copy()
df1_clean.columns = ['Code', 'Country', 'Rate_2023', 'Rate_2022', 'Male_Rate_2023', 'Male_Rate_2022', 
                    'Female_Rate_2023', 'Female_Rate_2022']

# Reshape from wide to long format for 2022 and 2023
df1_2022 = df1_clean[['Code', 'Country', 'Rate_2022', 'Male_Rate_2022', 'Female_Rate_2022']].copy()
df1_2022['Year'] = 2022
df1_2022.columns = ['Code', 'Country', 'Suicide_Rate', 'Male_Suicide_Rate', 'Female_Suicide_Rate', 'Year']

df1_2023 = df1_clean[['Code', 'Country', 'Rate_2023', 'Male_Rate_2023', 'Female_Rate_2023']].copy()
df1_2023['Year'] = 2023
df1_2023.columns = ['Code', 'Country', 'Suicide_Rate', 'Male_Suicide_Rate', 'Female_Suicide_Rate', 'Year']

# Combine 2022 and 2023 data
df1_long = pd.concat([df1_2022, df1_2023], ignore_index=True)

# Clean the second dataset (2011-2021 data)
df2_clean = df2.copy()
df2_clean.columns = ['Country', 'Code', 'Year', 'Suicide_Rate']

# Filter for years 2011-2021 only
df2_filtered = df2_clean[df2_clean['Year'].between(2011, 2021)]

# Merge the datasets
merged_df = pd.concat([df2_filtered, df1_long], ignore_index=True)

# Remove duplicates (keeping first occurrence)
merged_df = merged_df.drop_duplicates(subset=['Country', 'Code', 'Year'])

# Sort and clean
merged_df = merged_df.sort_values(['Country', 'Year']).reset_index(drop=True)

# Add numerical enhancements
merged_df = merged_df.sort_values(['Country', 'Year'])
merged_df['Yearly_Change'] = merged_df.groupby('Country')['Suicide_Rate'].transform(
    lambda x: x - x.shift(1)
)

merged_df['Yearly_Growth_Rate'] = merged_df.groupby('Country')['Suicide_Rate'].transform(
    lambda x: ((x - x.shift(1)) / x.shift(1).clip(lower=0.1)) * 100
)

# Calculate rolling averages
merged_df['Rolling_Avg_3yr'] = merged_df.groupby('Country')['Suicide_Rate'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Global comparative metrics
global_avg_by_year = merged_df.groupby('Year')['Suicide_Rate'].transform('mean')
merged_df['Deviation_From_Global_Avg'] = merged_df['Suicide_Rate'] - global_avg_by_year

merged_df['Global_Percentile_Rank'] = merged_df.groupby('Year')['Suicide_Rate'].transform(
    lambda x: x.rank(pct=True) * 100
)

# Suicide rate level flags (boolean)
merged_df['Very_High_Risk'] = merged_df['Suicide_Rate'] > 20
merged_df['High_Risk'] = (merged_df['Suicide_Rate'] > 10) & (merged_df['Suicide_Rate'] <= 20)
merged_df['Moderate_Risk'] = (merged_df['Suicide_Rate'] > 5) & (merged_df['Suicide_Rate'] <= 10)
merged_df['Low_Risk'] = merged_df['Suicide_Rate'] <= 5

# Display results
print("MERGED SUICIDE RATE DATASET")
print("=" * 45)
print(f"Shape: {merged_df.shape}")
print(f"Years: {merged_df['Year'].min()}-{merged_df['Year'].max()}")
print(f"Countries: {merged_df['Country'].nunique()}")
print(f"Total Records: {len(merged_df):,}")
print(f"Latest Global Avg: {merged_df[merged_df['Year'] == merged_df['Year'].max()]['Suicide_Rate'].mean():.1f} per 100K")

print("\nFirst 10 rows:")
print(merged_df[['Country', 'Year', 'Suicide_Rate', 'Yearly_Change', 'Global_Percentile_Rank']].head(10))

# Save merged dataset
merged_df.to_csv('merged_suicide_rate_data.csv', index=False)
print("\nMerged dataset saved as 'merged_suicide_rate_data.csv'")

MERGED SUICIDE RATE DATASET
Shape: (2619, 15)
Years: 2011-2023
Countries: 216
Total Records: 2,619
Latest Global Avg: 9.7 per 100K

First 10 rows:
       Country  Year  Suicide_Rate  Yearly_Change  Global_Percentile_Rank
0  Afghanistan  2011          3.67            NaN               15.920398
1  Afghanistan  2012          3.65          -0.02               16.169154
2  Afghanistan  2013          3.63          -0.02               15.422886
3  Afghanistan  2014          3.57          -0.06               15.422886
4  Afghanistan  2015          3.49          -0.08               15.920398
5  Afghanistan  2016          3.49           0.00               16.417910
6  Afghanistan  2017          3.58           0.09               18.407960
7  Afghanistan  2018          3.58           0.00               16.915423
8  Afghanistan  2019          3.55          -0.03               17.910448
9  Afghanistan  2020          3.52          -0.03               19.402985

Merged dataset saved as 'merged_suicid

### Merge Unemployement

In [1]:
import pandas as pd
import numpy as np

# Read all unemployment datasets
unemployment_df = pd.read_csv('Datasets/Uneployment/Uneployment.csv')
country_meta_df = pd.read_csv('Datasets/Uneployment/Metadata_Country_API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_254884.csv')
indicator_meta_df = pd.read_csv('Datasets/Uneployment/Metadata_Indicator_API_SL.UEM.TOTL.ZS_DS2_en_csv_v2_254884.csv')

# Display column names to debug
print("Unemployment DataFrame columns:", unemployment_df.columns.tolist())
print("Country Metadata columns:", country_meta_df.columns.tolist())
print("Indicator Metadata columns:", indicator_meta_df.columns.tolist())

# Clean the main unemployment dataset
unemp_clean = unemployment_df.copy()
unemp_clean = unemp_clean.dropna(how='all', axis=1)  # Remove empty columns

# Reshape from wide to long format
year_cols = [str(year) for year in range(2011, 2025)]
unemp_long = pd.melt(unemp_clean,
                    id_vars=['Country Name', 'Country Code'],
                    value_vars=year_cols,
                    var_name='Year',
                    value_name='Unemployment_Rate')

# Convert Year to integer and handle missing values
unemp_long['Year'] = unemp_long['Year'].astype(int)
unemp_long['Unemployment_Rate'] = pd.to_numeric(unemp_long['Unemployment_Rate'], errors='coerce')

# Clean country metadata column names (remove quotes and spaces)
country_meta_df.columns = country_meta_df.columns.str.replace('"', '').str.strip()
print("Cleaned Country Metadata columns:", country_meta_df.columns.tolist())

# Rename country metadata columns for merge
country_meta_df = country_meta_df.rename(columns={
    'Country Code': 'Country_Code',
    'Region': 'Region',
    'IncomeGroup': 'Income_Group',
    'SpecialNotes': 'Special_Notes',
    'TableName': 'Table_Name'
})

# Clean indicator metadata column names
indicator_meta_df.columns = indicator_meta_df.columns.str.replace('"', '').str.strip()

# Merge with country metadata using the correct column name
merged_df = pd.merge(
    unemp_long,
    country_meta_df,
    left_on='Country Code',
    right_on='Country_Code',
    how='left'
)

# Add indicator metadata information
merged_df['Indicator_Code'] = 'SL.UEM.TOTL.ZS'
merged_df['Indicator_Name'] = 'Unemployment, total (% of total labor force) (modeled ILO estimate)'
merged_df['Source_Organization'] = 'International Labour Organization (ILO)'

# Add numerical enhancements
merged_df = merged_df.sort_values(['Country Name', 'Year'])
merged_df['Yearly_Change'] = merged_df.groupby('Country Name')['Unemployment_Rate'].transform(
    lambda x: x - x.shift(1)
)

merged_df['Yearly_Growth_Rate'] = merged_df.groupby('Country Name')['Unemployment_Rate'].transform(
    lambda x: ((x - x.shift(1)) / x.shift(1).clip(lower=0.1)) * 100
)

# Calculate rolling averages
merged_df['Rolling_Avg_3yr'] = merged_df.groupby('Country Name')['Unemployment_Rate'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Global comparative metrics
global_avg_by_year = merged_df.groupby('Year')['Unemployment_Rate'].transform('mean')
merged_df['Deviation_From_Global_Avg'] = merged_df['Unemployment_Rate'] - global_avg_by_year

merged_df['Global_Percentile_Rank'] = merged_df.groupby('Year')['Unemployment_Rate'].transform(
    lambda x: x.rank(pct=True) * 100
)

# Unemployment level flags (boolean)
merged_df['Very_High_Unemployment'] = merged_df['Unemployment_Rate'] > 15
merged_df['High_Unemployment'] = (merged_df['Unemployment_Rate'] > 8) & (merged_df['Unemployment_Rate'] <= 15)
merged_df['Moderate_Unemployment'] = (merged_df['Unemployment_Rate'] > 5) & (merged_df['Unemployment_Rate'] <= 8)
merged_df['Low_Unemployment'] = merged_df['Unemployment_Rate'] <= 5

# Sort and clean final dataset
merged_df = merged_df.sort_values(['Country Name', 'Year']).reset_index(drop=True)

# Select and rename final columns
final_columns = {
    'Country Name': 'Country',
    'Country Code': 'Code',
    'Year': 'Year',
    'Unemployment_Rate': 'Unemployment_Rate',
    'Region': 'Region',
    'Income_Group': 'Income_Group',
    'Special_Notes': 'Special_Notes',
    'Indicator_Code': 'Indicator_Code',
    'Indicator_Name': 'Indicator_Name',
    'Source_Organization': 'Source_Organization',
    'Yearly_Change': 'Yearly_Change',
    'Yearly_Growth_Rate': 'Yearly_Growth_Rate',
    'Rolling_Avg_3yr': 'Rolling_Avg_3yr',
    'Deviation_From_Global_Avg': 'Deviation_From_Global_Avg',
    'Global_Percentile_Rank': 'Global_Percentile_Rank',
    'Very_High_Unemployment': 'Very_High_Unemployment',
    'High_Unemployment': 'High_Unemployment',
    'Moderate_Unemployment': 'Moderate_Unemployment',
    'Low_Unemployment': 'Low_Unemployment'
}

final_df = merged_df[list(final_columns.keys())].rename(columns=final_columns)

# Display results
print("\nMERGED UNEMPLOYMENT DATASET")
print("=" * 45)
print(f"Shape: {final_df.shape}")
print(f"Years: {final_df['Year'].min()}-{final_df['Year'].max()}")
print(f"Countries: {final_df['Country'].nunique()}")
print(f"Total Records: {len(final_df):,}")
latest_year = final_df['Year'].max()
latest_avg = final_df[final_df['Year'] == latest_year]['Unemployment_Rate'].mean()
print(f"Latest Global Avg ({latest_year}): {latest_avg:.1f}%")

print("\nFirst 10 rows:")
display_cols = ['Country', 'Year', 'Unemployment_Rate', 'Yearly_Change', 'Income_Group', 'Global_Percentile_Rank']
print(final_df[display_cols].head(10))

# Save merged dataset
final_df.to_csv('merged_unemployment_data.csv', index=False)
print("\nMerged dataset saved as 'merged_unemployment_data.csv'")

Unemployment DataFrame columns: ['Country Name', 'Country Code', 'Indicator Name', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Country Metadata columns: ['Country Code', 'Region', 'IncomeGroup', 'SpecialNotes', 'TableName', 'Unnamed: 5']
Indicator Metadata columns: ['INDICATOR_CODE', 'INDICATOR_NAME', 'SOURCE_NOTE', 'SOURCE_ORGANIZATION', 'Unnamed: 4']
Cleaned Country Metadata columns: ['Country Code', 'Region', 'IncomeGroup', 'SpecialNotes', 'TableName', 'Unnamed: 5']

MERGED UNEMPLOYMENT DATASET
Shape: (3724, 19)
Years: 2011-2024
Countries: 266
Total Records: 3,724
Latest Global Avg (2024): 6.5%

First 10 rows:
       Country  Year  Unemployment_Rate  Yearly_Change Income_Group  \
0  Afghanistan  2011              7.784            NaN   Low income   
1  Afghanistan  2012              7.856          0.072   Low income   
2  Afghanistan  2013              7.930          0.074   Low income   
3  Afghanistan  2014       

### Combine Urban Population

In [2]:
import pandas as pd
import numpy as np

# Read both urban population datasets
df1 = pd.read_csv('Datasets/Urban Population/cc5e5ebb-41d2-4886-81f1-0dbe718622ab_Series - Metadata.csv')
df2 = pd.read_csv('Datasets/Urban Population/Urban Population as percentage of total population - Urbanization as percentage of total population.csv')

# Clean and reshape the first dataset (with metadata)
df1_clean = df1.copy()
df1_clean = df1_clean[df1_clean['Series Name'] == 'Urban population (% of total population)']  # Keep only urban population data

# Reshape from wide to long format
year_cols = [col for col in df1_clean.columns if 'YR' in col]
df1_long = pd.melt(df1_clean, 
                  id_vars=['Country Name', 'Country Code'], 
                  value_vars=year_cols,
                  var_name='Year_Str', 
                  value_name='Urban_Population_Pct')

# Extract year from column names and convert to integer
df1_long['Year'] = df1_long['Year_Str'].str.extract(r'(\d+)').astype(int)
df1_long = df1_long.drop('Year_Str', axis=1)

# Convert Urban_Population_Pct to numeric
df1_long['Urban_Population_Pct'] = pd.to_numeric(df1_long['Urban_Population_Pct'], errors='coerce')

# Clean the second dataset
df2_clean = df2.copy()

# Reshape second dataset from wide to long format
year_cols_df2 = [col for col in df2_clean.columns if 'YR' in col]
df2_long = pd.melt(df2_clean,
                  id_vars=['Country Name', 'Country Code'],
                  value_vars=year_cols_df2,
                  var_name='Year_Str',
                  value_name='Urban_Population_Pct')

# Extract year from column names
df2_long['Year'] = df2_long['Year_Str'].str.extract(r'(\d+)').astype(int)
df2_long = df2_long.drop('Year_Str', axis=1)

# Convert Urban_Population_Pct to numeric
df2_long['Urban_Population_Pct'] = pd.to_numeric(df2_long['Urban_Population_Pct'], errors='coerce')

# Merge the datasets
merged_df = pd.concat([df1_long, df2_long], ignore_index=True)

# Remove duplicates (keeping first occurrence)
merged_df = merged_df.drop_duplicates(subset=['Country Name', 'Country Code', 'Year'])

# Remove rows with missing urban population values
merged_df = merged_df.dropna(subset=['Urban_Population_Pct'])

# Sort and clean
merged_df = merged_df.sort_values(['Country Name', 'Year']).reset_index(drop=True)
merged_df.columns = ['Country', 'Code', 'Urban_Population_Pct', 'Year']

# Add numerical enhancements
merged_df = merged_df.sort_values(['Country', 'Year'])
merged_df['Yearly_Change'] = merged_df.groupby('Country')['Urban_Population_Pct'].transform(
    lambda x: x - x.shift(1)
)

merged_df['Yearly_Growth_Rate'] = merged_df.groupby('Country')['Urban_Population_Pct'].transform(
    lambda x: ((x - x.shift(1)) / x.shift(1).clip(lower=0.1)) * 100
)

# Calculate multi-year urbanization changes
merged_df['Change_5yr'] = merged_df.groupby('Country')['Urban_Population_Pct'].transform(
    lambda x: x - x.shift(5)
)

merged_df['Change_10yr'] = merged_df.groupby('Country')['Urban_Population_Pct'].transform(
    lambda x: x - x.shift(10)
)

# Calculate rolling averages
merged_df['Rolling_Avg_3yr'] = merged_df.groupby('Country')['Urban_Population_Pct'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

# Global comparative metrics
global_avg_by_year = merged_df.groupby('Year')['Urban_Population_Pct'].transform('mean')
merged_df['Deviation_From_Global_Avg'] = merged_df['Urban_Population_Pct'] - global_avg_by_year

merged_df['Global_Percentile_Rank'] = merged_df.groupby('Year')['Urban_Population_Pct'].transform(
    lambda x: x.rank(pct=True) * 100
)

# Urbanization level flags (boolean)
merged_df['Highly_Urbanized'] = merged_df['Urban_Population_Pct'] > 75
merged_df['Moderately_Urbanized'] = (merged_df['Urban_Population_Pct'] > 50) & (merged_df['Urban_Population_Pct'] <= 75)
merged_df['Low_Urbanized'] = (merged_df['Urban_Population_Pct'] > 25) & (merged_df['Urban_Population_Pct'] <= 50)
merged_df['Very_Low_Urbanized'] = merged_df['Urban_Population_Pct'] <= 25
merged_df['Fully_Urbanized'] = merged_df['Urban_Population_Pct'] >= 95

# Calculate urbanization acceleration
merged_df['Urbanization_Acceleration'] = merged_df.groupby('Country')['Yearly_Change'].transform(
    lambda x: x - x.shift(1)
)

# Display results
print("MERGED URBAN POPULATION DATASET")
print("=" * 45)
print(f"Shape: {merged_df.shape}")
print(f"Years: {merged_df['Year'].min()}-{merged_df['Year'].max()}")
print(f"Countries: {merged_df['Country'].nunique()}")
print(f"Total Records: {len(merged_df):,}")
latest_year = merged_df['Year'].max()
latest_avg = merged_df[merged_df['Year'] == latest_year]['Urban_Population_Pct'].mean()
print(f"Latest Global Urbanization Avg ({latest_year}): {latest_avg:.1f}%")

print("\nFirst 10 rows:")
print(merged_df[['Country', 'Year', 'Urban_Population_Pct', 'Yearly_Change', 'Global_Percentile_Rank']].head(10))

# Save merged dataset
merged_df.to_csv('merged_urban_population_data.csv', index=False)
print("\nMerged dataset saved as 'merged_urban_population_data.csv'")

MERGED URBAN POPULATION DATASET
Shape: (3682, 17)
Years: 2011-2024
Countries: 263
Total Records: 3,682
Latest Global Urbanization Avg (2024): 61.6%

First 10 rows:
       Country  Year  Urban_Population_Pct  Yearly_Change  \
0  Afghanistan  2011                23.948            NaN   
1  Afghanistan  2012                24.160          0.212   
2  Afghanistan  2013                24.373          0.213   
3  Afghanistan  2014                24.587          0.214   
4  Afghanistan  2015                24.803          0.216   
5  Afghanistan  2016                25.020          0.217   
6  Afghanistan  2017                25.250          0.230   
7  Afghanistan  2018                25.495          0.245   
8  Afghanistan  2019                25.754          0.259   
9  Afghanistan  2020                26.026          0.272   

   Global_Percentile_Rank  
0                7.604563  
1                7.604563  
2                7.604563  
3                7.604563  
4                7.60456

## Processing Into Singular Dataset

### Alcohol and Corruption

In [3]:
import pandas as pd
import numpy as np

def process_alcohol_data(file_path):
    """Process alcohol consumption dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data only and our target years
    alcohol_df = df[
        (df['DIM_GEO_CODE_TYPE'] == 'COUNTRY') & 
        (df['DIM_TIME'].between(2011, 2024))
    ].copy()
    
    # Select and rename columns
    alcohol_df = alcohol_df[['GEO_NAME_SHORT', 'DIM_TIME', 'RATE_PER_CAPITA_N']]
    alcohol_df.columns = ['country', 'year', 'alcohol_consumption']
    
    return alcohol_df

def process_corruption_data(file_path):
    """Process corruption index dataset from wide to long format"""
    df = pd.read_csv(file_path)
    
    # Melt the wide format years into long format
    year_columns = [col for col in df.columns if 'CPI score' in col]
    
    # Extract years from column names
    years = [int(col.split(' ')[-1]) for col in year_columns]
    
    # Create long format
    corruption_long = pd.melt(
        df, 
        id_vars=['Country / Territory'],
        value_vars=year_columns,
        var_name='year_str',
        value_name='corruption_index'
    )
    
    # Extract year from column name and filter
    corruption_long['year'] = corruption_long['year_str'].str.extract('(\d+)').astype(int)
    corruption_long = corruption_long[corruption_long['year'].between(2011, 2024)]
    
    # Clean up and rename
    corruption_long = corruption_long[['Country / Territory', 'year', 'corruption_index']]
    corruption_long.columns = ['country', 'year', 'corruption_index']
    
    return corruption_long

# Process both datasets
print("Processing datasets...")
alcohol_df = process_alcohol_data('Alcohol Consumption.csv')
corruption_df = process_corruption_data('Corruption Index.csv')

print("Alcohol Data Sample:")
print(alcohol_df.head())
print(f"\nAlcohol data shape: {alcohol_df.shape}")

print("\nCorruption Data Sample:")
print(corruption_df.head())
print(f"\nCorruption data shape: {corruption_df.shape}")

# Merge the datasets
merged_df = pd.merge(alcohol_df, corruption_df, on=['country', 'year'], how='outer')

print(f"\nMerged dataset shape: {merged_df.shape}")
print("\nMerged Data Sample:")
print(merged_df.head(10))

# Check for missing values
print(f"\nMissing values by column:")
print(merged_df.isnull().sum())

# SAVE THE MERGED DATASET
output_file = 'merged/merged_alcohol_corruption_2011_2024.csv'
merged_df.to_csv(output_file, index=False)
print(f"\nMERGED DATASET SAVED AS: {output_file}")

# Summary statistics
print(f"\n📊 FINAL SUMMARY:")
print(f"Total merged records: {len(merged_df)}")
print(f"Complete records (no missing data): {len(complete_cases_df)}")
print(f"Countries in merged data: {merged_df['country'].nunique()}")
print(f"Years covered: {sorted(merged_df['year'].unique())}")

Processing datasets...
Alcohol Data Sample:
        country  year  alcohol_consumption
176      Latvia  2014            12.399305
177     Liberia  2014             2.739028
179       Libya  2014             0.015712
180   Lithuania  2014            14.412071
181  Luxembourg  2014            11.736241

Alcohol data shape: (2256, 3)

Corruption Data Sample:
       country  year  corruption_index
0  Afghanistan  2024              17.0
1      Albania  2024              42.0
2      Algeria  2024              34.0
3       Angola  2024              32.0
4    Argentina  2024              37.0

Corruption data shape: (1991, 3)

Merged dataset shape: (2798, 4)

Merged Data Sample:
       country  year  alcohol_consumption  corruption_index
0  Afghanistan  2011             0.027563               NaN
1  Afghanistan  2012             0.028278               NaN
2  Afghanistan  2013             0.026020               NaN
3  Afghanistan  2014             0.022397              12.0
4  Afghanistan  2015

<>:40: SyntaxWarning: invalid escape sequence '\d'
<>:40: SyntaxWarning: invalid escape sequence '\d'
C:\Users\ibrahim\AppData\Local\Temp\ipykernel_9340\1371372401.py:40: SyntaxWarning: invalid escape sequence '\d'
  corruption_long['year'] = corruption_long['year_str'].str.extract('(\d+)').astype(int)


### Merge Education Expenditure, Happiness Data, Obesity Data

In [4]:
import pandas as pd
import numpy as np

def load_existing_merged_data():
    """Load our previously merged alcohol-corruption dataset"""
    try:
        return pd.read_csv('merged_alcohol_corruption_2011_2024.csv')
    except FileNotFoundError:
        print("Existing merged file not found. Starting fresh...")
        return pd.DataFrame()

def process_education_data(file_path):
    """Process education expenditure dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    education_df = df[
        (~df['Country'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Handle missing values (represented as '..')
    education_df['Value_ds1'] = pd.to_numeric(education_df['Value_ds1'], errors='coerce')
    education_df['Value_ds2'] = pd.to_numeric(education_df['Value_ds2'], errors='coerce')
    
    # Select and rename columns - use Value_ds1 as primary
    education_df = education_df[['Country', 'Year', 'Value_ds1']]
    education_df.columns = ['country', 'year', 'education_expenditure_pct_gdp']
    
    # Drop rows with missing values
    education_df = education_df.dropna(subset=['education_expenditure_pct_gdp'])
    
    return education_df

def process_happiness_data(file_path):
    """Process enhanced happiness dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for our target years and countries with codes
    happiness_df = df[
        (df['Year'].between(2011, 2024)) & 
        (df['Code'].notna())
    ].copy()
    
    # Select key columns
    happiness_cols = ['Entity', 'Year', 'Happiness_Score', 'Yearly_Change', 
                     'Global_Percentile_Rank', 'Very_High_Happiness', 'Very_Low_Happiness']
    
    happiness_df = happiness_df[happiness_cols]
    happiness_df.columns = ['country', 'year', 'happiness_score', 'happiness_yearly_change',
                          'happiness_global_percentile', 'very_high_happiness', 'very_low_happiness']
    
    return happiness_df

def process_obesity_data(file_path):
    """Process enhanced obesity dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for our target years and countries with codes
    obesity_df = df[
        (df['Year'].between(2011, 2024)) & 
        (df['Code'].notna())
    ].copy()
    
    # Select key columns
    obesity_cols = ['Entity', 'Year', 'Overweight_Pct', 'Yearly_Change', 
                   'Global_Percentile_Rank', 'High_Obesity', 'Very_High_Obesity']
    
    obesity_df = obesity_df[obesity_cols]
    obesity_df.columns = ['country', 'year', 'overweight_pct', 'obesity_yearly_change',
                        'obesity_global_percentile', 'high_obesity', 'very_high_obesity']
    
    return obesity_df

# Load existing merged data
print("Loading existing merged dataset...")
master_df = load_existing_merged_data()

if not master_df.empty:
    print(f"Existing merged data shape: {master_df.shape}")
else:
    print("Starting with empty master dataset")
    master_df = pd.DataFrame()

# Process new datasets
print("\nProcessing new datasets...")

# Education expenditure
print("Processing Education Expenditure...")
education_df = process_education_data('Education Expenditure.csv')
print(f"Education data shape: {education_df.shape}")

# Happiness data
print("Processing Happiness Data...")
happiness_df = process_happiness_data('enhanced_happiness_data.csv')
print(f"Happiness data shape: {happiness_df.shape}")

# Obesity data
print("Processing Obesity Data...")
obesity_df = process_obesity_data('enhanced_obesity_data.csv')
print(f"Obesity data shape: {obesity_df.shape}")

# Merge all datasets step by step
print("\nMerging all datasets...")

if master_df.empty:
    # Start with education data as base
    master_df = education_df
    print("Starting merge with Education data as base")
else:
    # Merge with existing master
    master_df = pd.merge(master_df, education_df, on=['country', 'year'], how='outer')
    print("Merged with Education data")

# Merge happiness data
master_df = pd.merge(master_df, happiness_df, on=['country', 'year'], how='outer')
print("Merged with Happiness data")

# Merge obesity data
master_df = pd.merge(master_df, obesity_df, on=['country', 'year'], how='outer')
print("Merged with Obesity data")

print(f"\nFinal merged dataset shape: {master_df.shape}")

# Data quality checks
print("\n📊 DATA QUALITY REPORT:")
print(f"Total records: {len(master_df)}")
print(f"Countries: {master_df['country'].nunique()}")
print(f"Years: {sorted(master_df['year'].dropna().unique())}")

print("\nMissing values by column:")
missing_data = master_df.isnull().sum()
missing_pct = (master_df.isnull().sum() / len(master_df)) * 100
missing_report = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
})
print(missing_report[missing_report['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False))

# Save the comprehensive merged dataset
output_file = 'comprehensive_merged_dataset_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"\nCOMPREHENSIVE MERGED DATASET SAVED AS: {output_file}")

# Also save a version with country-year index for easy analysis
master_df_with_index = master_df.copy()
master_df_with_index['country_year'] = master_df_with_index['country'] + '_' + master_df_with_index['year'].astype(str)
master_df_with_index.to_csv('merged/comprehensive_merged_dataset_indexed.csv', index=False)
print("INDEXED VERSION SAVED AS: comprehensive_merged_dataset_indexed.csv")


# Final summary
print(f"\nFINAL MERGE COMPLETE!")
print(f"Total datasets merged: 5")
print(f"Final columns: {len(master_df.columns)}")
print("Columns:", list(master_df.columns))
print(f"Time range: 2011-2024")
print(f"Countries in final dataset: {master_df['country'].nunique()}")

Loading existing merged dataset...
Existing merged file not found. Starting fresh...
Starting with empty master dataset

Processing new datasets...
Processing Education Expenditure...
Education data shape: (2544, 3)
Processing Happiness Data...
Happiness data shape: (1982, 7)
Processing Obesity Data...
Obesity data shape: (2400, 7)

Merging all datasets...
Starting merge with Education data as base
Merged with Happiness data
Merged with Obesity data

Final merged dataset shape: (3592, 13)

📊 DATA QUALITY REPORT:
Total records: 3592
Countries: 277
Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values by column:
                               Missing_Count  Missing_Percentage
happiness_yearly_change                 1778           49.498886
happiness_score                         1610           44.821826
happiness_global_percentile             1610           44.821826
very_high_happiness                     1610           44.821826
ver

### Merge Poverty, Fertility Rate, Consumption Per Capita

In [ ]:
import pandas as pd
import numpy as np

def load_existing_comprehensive_data():
    """Load our existing comprehensive merged dataset"""
    try:
        return pd.read_csv('comprehensive_merged_dataset_2011_2024.csv')
    except FileNotFoundError:
        print("Existing comprehensive file not found. Starting fresh...")
        return pd.DataFrame()

def process_poverty_data(file_path):
    """Process enhanced poverty dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for our target years 2011-2024
    poverty_df = df[df['Year'].between(2011, 2024)].copy()
    
    # Select key columns - focus on core poverty metrics
    poverty_cols = ['Country', 'Year', 'Poverty_Rate', 'Yearly_Change', 
                   'Global_Percentile_Rank', 'Extreme_Poverty', 'Moderate_Poverty',
                   'Low_Poverty', 'Zero_Poverty']
    
    poverty_df = poverty_df[poverty_cols]
    poverty_df.columns = ['country', 'year', 'poverty_rate', 'poverty_yearly_change',
                        'poverty_global_percentile', 'extreme_poverty', 'moderate_poverty',
                        'low_poverty', 'zero_poverty']
    
    return poverty_df

def process_fertility_data(file_path):
    """Process fertility rate dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    fertility_df = df[
        (~df['Country Name'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Handle missing values (represented as '..') - use Fertility_Rate_data as primary
    fertility_df['Fertility_Rate_data'] = pd.to_numeric(fertility_df['Fertility_Rate_data'], errors='coerce')
    
    # Select and rename columns
    fertility_df = fertility_df[['Country Name', 'Year', 'Fertility_Rate_data']]
    fertility_df.columns = ['country', 'year', 'fertility_rate']
    
    # Drop rows with missing fertility data
    fertility_df = fertility_df.dropna(subset=['fertility_rate'])
    
    return fertility_df

def process_consumption_data(file_path):
    """Process final consumption per capita dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    consumption_df = df[
        (~df['Country Name'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Handle missing values (represented as '..') - use Consumption_Per_Capita_data as primary
    consumption_df['Consumption_Per_Capita_data'] = pd.to_numeric(
        consumption_df['Consumption_Per_Capita_data'], errors='coerce'
    )
    
    # Select and rename columns
    consumption_df = consumption_df[['Country Name', 'Year', 'Consumption_Per_Capita_data']]
    consumption_df.columns = ['country', 'year', 'consumption_per_capita_usd']
    
    # Drop rows with missing consumption data
    consumption_df = consumption_df.dropna(subset=['consumption_per_capita_usd'])
    
    return consumption_df

# Load existing comprehensive data
print("Loading existing comprehensive dataset...")
master_df = load_existing_comprehensive_data()

if not master_df.empty:
    print(f"Existing comprehensive data shape: {master_df.shape}")
    print(f"Existing columns: {len(master_df.columns)}")
else:
    print("Starting with empty master dataset")
    master_df = pd.DataFrame()

# Process new datasets
print("\nProcessing new datasets...")

# Poverty data
print("Processing Poverty Data...")
poverty_df = process_poverty_data('enhanced_poverty_data.csv')
print(f"Poverty data shape: {poverty_df.shape}")
print(f"Poverty years: {sorted(poverty_df['year'].unique())}")

# Fertility data
print("Processing Fertility Data...")
fertility_df = process_fertility_data('Fertility_Rate_Merged.csv')
print(f"Fertility data shape: {fertility_df.shape}")
print(f"Fertility years: {sorted(fertility_df['year'].unique())}")

# Consumption data
print("Processing Consumption Data...")
consumption_df = process_consumption_data('Final_Consumption_Per_Capita_Merged.csv')
print(f"Consumption data shape: {consumption_df.shape}")
print(f"Consumption years: {sorted(consumption_df['year'].unique())}")

# Merge all datasets step by step
print("\nMerging all datasets...")

if master_df.empty:
    # Start with poverty data as base if no existing data
    master_df = poverty_df
    print("Starting merge with Poverty data as base")
else:
    # Merge with existing master
    master_df = pd.merge(master_df, poverty_df, on=['country', 'year'], how='outer')
    print("Merged with Poverty data")

# Merge fertility data
master_df = pd.merge(master_df, fertility_df, on=['country', 'year'], how='outer')
print("Merged with Fertility data")

# Merge consumption data
master_df = pd.merge(master_df, consumption_df, on=['country', 'year'], how='outer')
print("Merged with Consumption data")

print(f"\nFinal merged dataset shape: {master_df.shape}")

# Data quality checks
print("\n📊 DATA QUALITY REPORT:")
print(f"Total records: {len(master_df)}")
print(f"Countries: {master_df['country'].nunique()}")
print(f"Years: {sorted(master_df['year'].dropna().unique())}")

print("\nMissing values by column:")
missing_data = master_df.isnull().sum()
missing_pct = (master_df.isnull().sum() / len(master_df)) * 100
missing_report = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
})
missing_summary = missing_report[missing_report['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False)
print(missing_summary.head(10))  # Show top 10 columns with most missing data

# Calculate coverage statistics
print("\n📈 DATA COVERAGE STATISTICS:")
total_country_years = len(master_df)
coverage_stats = {}

for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct

coverage_df = pd.DataFrame({
    'Column': list(coverage_stats.keys()),
    'Coverage_Percentage': list(coverage_stats.values())
}).sort_values('Coverage_Percentage', ascending=False)

print(coverage_df.head(15))

# Save the updated comprehensive merged dataset
output_file = 'merged/comprehensive_poverty_fertility_consumptionPerCapita_2011_2024_v2.csv'
master_df.to_csv(output_file, index=False)
print(f"\n✅ UPDATED COMPREHENSIVE MERGED DATASET SAVED AS: {output_file}")

# Final summary
print(f"\n🎯 MERGE COMPLETE!")
print(f"Total datasets merged so far: 8")
print(f"Final columns: {len(master_df.columns)}")
print(f"Time range: 2011-2024")
print(f"Countries in final dataset: {master_df['country'].nunique()}")
print(f"Total country-year observations: {len(master_df)}")

# Show sample of the merged data
print("\n📋 SAMPLE OF MERGED DATA:")
print(master_df.head(10))

Loading existing comprehensive dataset...
Existing comprehensive data shape: (3592, 13)
Existing columns: 13

Processing new datasets...
Processing Poverty Data...
Poverty data shape: (1220, 9)
Poverty years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Processing Fertility Data...
Fertility data shape: (2795, 3)
Fertility years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Processing Consumption Data...
Consumption data shape: (2246, 3)
Consumption years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Merging all datasets...
Merged with Poverty data
Merged with Fertility data
Merged with Consumption data

Final merged dataset shape: (4090, 22)

📊 DATA QUALITY REPORT:
Total records: 4090
Countries: 304
Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values by column:
                             Missing_Count  Missing_Perc

### Merge GDP Happiness, Health Expenditure, Homicide Rate

In [7]:
import pandas as pd
import numpy as np

def process_gdp_happiness_data(file_path):
    """Process GDP and Happiness dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for our target years 2011-2024
    gdp_df = df[df['Year'].between(2011, 2024)].copy()
    
    # Select and rename columns
    gdp_df = gdp_df[['Country', 'Year', 'Life_Satisfaction', 'GDP_per_capita_PPP', 'Region']]
    gdp_df.columns = ['country', 'year', 'life_satisfaction', 'gdp_per_capita_ppp', 'region']
    
    # Handle missing values
    gdp_df['life_satisfaction'] = pd.to_numeric(gdp_df['life_satisfaction'], errors='coerce')
    gdp_df['gdp_per_capita_ppp'] = pd.to_numeric(gdp_df['gdp_per_capita_ppp'], errors='coerce')
    
    return gdp_df

def process_health_expenditure_data(file_path):
    """Process health expenditure dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    health_df = df[
        (~df['Country Name'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Handle missing values (represented as '..') - use Health_Expenditure_Per_Capita_data as primary
    health_df['Health_Expenditure_Per_Capita_data'] = pd.to_numeric(
        health_df['Health_Expenditure_Per_Capita_data'], errors='coerce'
    )
    
    # Select and rename columns
    health_df = health_df[['Country Name', 'Year', 'Health_Expenditure_Per_Capita_data']]
    health_df.columns = ['country', 'year', 'health_expenditure_per_capita_usd']
    
    return health_df

def process_homicide_data(file_path):
    """Process homicide rate dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    homicide_df = df[
        (~df['Country Name'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Handle missing values - use Homicide_Rate_data as primary
    homicide_df['Homicide_Rate_data'] = pd.to_numeric(
        homicide_df['Homicide_Rate_data'], errors='coerce'
    )
    
    # Select and rename columns
    homicide_df = homicide_df[['Country Name', 'Year', 'Homicide_Rate_data']]
    homicide_df.columns = ['country', 'year', 'homicide_rate_per_100k']
    
    return homicide_df

# Process all three datasets
print("Processing the three datasets...")

# GDP and Happiness data
print("1. Processing GDP_Happiness_Panel.csv...")
gdp_happiness_df = process_gdp_happiness_data('GDP_Happiness_Panel.csv')
print(f"   GDP/Happiness data shape: {gdp_happiness_df.shape}")
print(f"   Years: {sorted(gdp_happiness_df['year'].unique())}")

# Health Expenditure data
print("2. Processing Health_Expenditure_Per_Capita_Merged.csv...")
health_df = process_health_expenditure_data('Health_Expenditure_Per_Capita_Merged.csv')
print(f"   Health expenditure data shape: {health_df.shape}")
print(f"   Years: {sorted(health_df['year'].unique())}")

# Homicide Rate data
print("3. Processing Homicide_Rate_Merged.csv...")
homicide_df = process_homicide_data('Homicide_Rate_Merged.csv')
print(f"   Homicide rate data shape: {homicide_df.shape}")
print(f"   Years: {sorted(homicide_df['year'].unique())}")

# Merge all three datasets step by step
print("\nMerging all three datasets...")

# Start with GDP/Happiness as base (it has region information)
master_df = gdp_happiness_df
print(f"Starting with GDP/Happiness data as base: {master_df.shape}")

# Merge with Health Expenditure data
master_df = pd.merge(master_df, health_df, on=['country', 'year'], how='outer')
print(f"After merging Health Expenditure: {master_df.shape}")

# Merge with Homicide Rate data
master_df = pd.merge(master_df, homicide_df, on=['country', 'year'], how='outer')
print(f"After merging Homicide Rate: {master_df.shape}")

# Data quality checks
print("\n📊 DATA QUALITY REPORT:")
print(f"Total records: {len(master_df)}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Years covered: {sorted(master_df['year'].dropna().unique())}")

print("\nMissing values by column:")
missing_data = master_df.isnull().sum()
missing_pct = (master_df.isnull().sum() / len(master_df)) * 100
missing_report = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
})
print(missing_report.sort_values('Missing_Percentage', ascending=False))

# Calculate coverage statistics
print("\n📈 DATA COVERAGE STATISTICS:")
total_country_years = len(master_df)
coverage_stats = {}

for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct

coverage_df = pd.DataFrame({
    'Column': list(coverage_stats.keys()),
    'Coverage_Percentage': list(coverage_stats.values())
}).sort_values('Coverage_Percentage', ascending=False)

print(coverage_df)

# Save the merged dataset
output_file = 'merged/merged_gdp_health_homicide_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"\nMERGED DATASET SAVED AS: {output_file}")

# Final summary
print(f"\n🎯 MERGE COMPLETE!")
print(f"Total datasets merged: 3")
print(f"Final columns: {len(master_df.columns)}")
print("Columns in final dataset:")
for col in master_df.columns:
    non_missing = master_df[col].notna().sum()
    pct = (non_missing / len(master_df)) * 100
    print(f"  - {col}: {non_missing}/{len(master_df)} ({pct:.1f}%)")

print(f"\nTime range: 2011-2024")
print(f"Countries in final dataset: {master_df['country'].nunique()}")
print(f"Total country-year observations: {len(master_df)}")

# Show sample of the merged data
print("\n📋 SAMPLE OF MERGED DATA (first 10 rows):")
print(master_df.head(10))

# Show regions available
print(f"\n🌍 REGIONS IN DATASET:")
if 'region' in master_df.columns:
    regions = master_df['region'].dropna().unique()
    print(f"Regions: {list(regions)}")
    print(f"Number of countries with region data: {master_df['region'].notna().sum()}")

Processing the three datasets...
1. Processing GDP_Happiness_Panel.csv...
   GDP/Happiness data shape: (2900, 5)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
2. Processing Health_Expenditure_Per_Capita_Merged.csv...
   Health expenditure data shape: (3682, 3)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
3. Processing Homicide_Rate_Merged.csv...
   Homicide rate data shape: (3668, 3)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Merging all three datasets...
Starting with GDP/Happiness data as base: (2900, 5)
After merging Health Expenditure: (4179, 6)
After merging Homicide Rate: (4459, 7)

📊 DATA QUALITY REPORT:
Total records: 4459
Unique countries: 334
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values by column:
                                   Missing_Count  Missing_Pe

### Merge Inflation, Internet Users, Population

In [9]:
import pandas as pd
import numpy as np

def process_inflation_data(file_path):
    """Process improved inflation dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for our target years 2011-2024
    inflation_df = df[df['Year'].between(2011, 2024)].copy()
    
    # Select key columns
    inflation_cols = ['Entity', 'Year', 'Inflation_Consumer_Prices_Annual_Pct', 
                     'Inflation_Category', 'Inflation_3yr_Avg', 'YoY_Increase']
    
    inflation_df = inflation_df[inflation_cols]
    inflation_df.columns = ['country', 'year', 'inflation_pct', 'inflation_category',
                          'inflation_3yr_avg', 'inflation_yoy_increase']
    
    return inflation_df

def process_internet_data(file_path):
    """Process internet users dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    internet_df = df[
        (~df['Country'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Handle missing values in Internet_Users_Pct
    internet_df['Internet_Users_Pct'] = pd.to_numeric(internet_df['Internet_Users_Pct'], errors='coerce')
    
    # Select key columns
    internet_cols = ['Country', 'Year', 'Internet_Users_Pct', 'Region', 'Income_Group']
    
    internet_df = internet_df[internet_cols]
    internet_df.columns = ['country', 'year', 'internet_users_pct', 'region', 'income_group']
    
    return internet_df

def process_population_data(file_path):
    """Process population dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    population_df = df[
        (~df['Country'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Select key columns
    population_cols = ['Country', 'Year', 'Population', 'Yearly_Change', 
                      'Yearly_Growth_Rate', 'Population_Millions']
    
    population_df = population_df[population_cols]
    population_df.columns = ['country', 'year', 'population', 'population_yearly_change',
                           'population_growth_rate', 'population_millions']
    
    return population_df

# Process all three datasets
print("Processing the three datasets...")

# Inflation data
print("1. Processing improved_inflation_data.csv...")
inflation_df = process_inflation_data('improved_inflation_data.csv')
print(f"   Inflation data shape: {inflation_df.shape}")
print(f"   Years: {sorted(inflation_df['year'].unique())}")

# Internet data
print("2. Processing merged_internet_users_data.csv...")
internet_df = process_internet_data('merged_internet_users_data.csv')
print(f"   Internet data shape: {internet_df.shape}")
print(f"   Years: {sorted(internet_df['year'].unique())}")

# Population data
print("3. Processing merged_population_data.csv...")
population_df = process_population_data('merged_population_data.csv')
print(f"   Population data shape: {population_df.shape}")
print(f"   Years: {sorted(population_df['year'].unique())}")

# Merge all three datasets step by step
print("\nMerging all three datasets...")

# Start with inflation data as base
master_df = inflation_df
print(f"Starting with Inflation data as base: {master_df.shape}")

# Merge with Internet data
master_df = pd.merge(master_df, internet_df, on=['country', 'year'], how='outer')
print(f"After merging Internet data: {master_df.shape}")

# Merge with Population data
master_df = pd.merge(master_df, population_df, on=['country', 'year'], how='outer')
print(f"After merging Population data: {master_df.shape}")

# Data quality checks
print("\n📊 DATA QUALITY REPORT:")
print(f"Total records: {len(master_df)}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Years covered: {sorted(master_df['year'].dropna().unique())}")

print("\nMissing values by column:")
missing_data = master_df.isnull().sum()
missing_pct = (master_df.isnull().sum() / len(master_df)) * 100
missing_report = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
})
print(missing_report.sort_values('Missing_Percentage', ascending=False))

# Calculate coverage statistics
print("\n📈 DATA COVERAGE STATISTICS:")
total_country_years = len(master_df)
coverage_stats = {}

for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct

coverage_df = pd.DataFrame({
    'Column': list(coverage_stats.keys()),
    'Coverage_Percentage': list(coverage_stats.values())
}).sort_values('Coverage_Percentage', ascending=False)

print(coverage_df)

# Save the merged dataset
output_file = 'merged/merged_inflation_internet_population_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"\nMERGED DATASET SAVED AS: {output_file}")


# Save a filtered version with reasonable data coverage (at least 30% coverage)
min_coverage = 30
columns_to_keep = ['country', 'year'] + coverage_df[coverage_df['Coverage_Percentage'] >= min_coverage]['Column'].tolist()


# Final summary
print(f"\n🎯 MERGE COMPLETE!")
print(f"Total datasets merged: 3")
print(f"Final columns: {len(master_df.columns)}")
print("Columns in final dataset:")
for col in master_df.columns:
    non_missing = master_df[col].notna().sum()
    pct = (non_missing / len(master_df)) * 100
    print(f"  - {col}: {non_missing}/{len(master_df)} ({pct:.1f}%)")

print(f"\nTime range: 2011-2024")
print(f"Countries in final dataset: {master_df['country'].nunique()}")
print(f"Total country-year observations: {len(master_df)}")

# Show sample of the merged data
print("\n📋 SAMPLE OF MERGED DATA (first 10 rows):")
print(master_df.head(10))

# Show regions and income groups available
print(f"\n🌍 REGIONS IN DATASET:")
if 'region' in master_df.columns:
    regions = master_df['region'].dropna().unique()
    print(f"Regions: {list(regions)}")
    print(f"Number of countries with region data: {master_df['region'].notna().sum()}")

print(f"\n💰 INCOME GROUPS IN DATASET:")
if 'income_group' in master_df.columns:
    income_groups = master_df['income_group'].dropna().unique()
    print(f"Income groups: {list(income_groups)}")
    print(f"Number of countries with income group data: {master_df['income_group'].notna().sum()}")

Processing the three datasets...
1. Processing improved_inflation_data.csv...
   Inflation data shape: (2554, 6)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
2. Processing merged_internet_users_data.csv...
   Internet data shape: (3563, 5)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
3. Processing merged_population_data.csv...
   Population data shape: (3570, 6)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Merging all three datasets...
Starting with Inflation data as base: (2554, 6)
After merging Internet data: (3932, 9)
After merging Population data: (3936, 13)

📊 DATA QUALITY REPORT:
Total records: 3936
Unique countries: 284
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values by column:
                          Missing_Count  Missing_Percentage
inflation_pct              

### Merge Suicide Rate, Unemployement, Urban Population

In [ ]:
import pandas as pd
import numpy as np

def process_suicide_data(file_path):
    """Process suicide rate dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for our target years 2011-2024 and country-level data
    suicide_df = df[
        (~df['Country'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Select key columns
    suicide_cols = ['Country', 'Year', 'Suicide_Rate', 'Male_Suicide_Rate', 'Female_Suicide_Rate',
                   'Global_Percentile_Rank', 'Very_High_Risk', 'High_Risk', 'Moderate_Risk', 'Low_Risk']
    
    suicide_df = suicide_df[suicide_cols]
    suicide_df.columns = ['country', 'year', 'suicide_rate', 'male_suicide_rate', 'female_suicide_rate',
                        'suicide_global_percentile', 'suicide_very_high_risk', 'suicide_high_risk',
                        'suicide_moderate_risk', 'suicide_low_risk']
    
    return suicide_df

def process_unemployment_data(file_path):
    """Process unemployment dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    unemployment_df = df[
        (~df['Country'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Select key columns
    unemployment_cols = ['Country', 'Year', 'Unemployment_Rate', 'Region', 'Income_Group',
                        'Yearly_Change', 'Yearly_Growth_Rate', 'Global_Percentile_Rank',
                        'Very_High_Unemployment', 'High_Unemployment', 'Moderate_Unemployment', 'Low_Unemployment']
    
    unemployment_df = unemployment_df[unemployment_cols]
    unemployment_df.columns = ['country', 'year', 'unemployment_rate', 'region', 'income_group',
                             'unemployment_yearly_change', 'unemployment_growth_rate', 'unemployment_global_percentile',
                             'unemployment_very_high', 'unemployment_high', 'unemployment_moderate', 'unemployment_low']
    
    return unemployment_df

def process_urban_population_data(file_path):
    """Process urban population dataset"""
    df = pd.read_csv(file_path)
    
    # Filter for country-level data (exclude regional aggregates) and our target years
    urban_df = df[
        (~df['Country'].str.contains('Africa', na=False)) & 
        (df['Year'].between(2011, 2024))
    ].copy()
    
    # Select key columns
    urban_cols = ['Country', 'Year', 'Urban_Population_Pct', 'Yearly_Change', 'Yearly_Growth_Rate',
                 'Global_Percentile_Rank', 'Highly_Urbanized', 'Moderately_Urbanized', 
                 'Low_Urbanized', 'Very_Low_Urbanized', 'Fully_Urbanized']
    
    urban_df = urban_df[urban_cols]
    urban_df.columns = ['country', 'year', 'urban_population_pct', 'urban_yearly_change', 'urban_growth_rate',
                      'urban_global_percentile', 'highly_urbanized', 'moderately_urbanized',
                      'low_urbanized', 'very_low_urbanized', 'fully_urbanized']
    
    return urban_df

# Process all three datasets
print("Processing the three datasets...")

# Suicide data
print("1. Processing merged_suicide_rate_data.csv...")
suicide_df = process_suicide_data('merged_suicide_rate_data.csv')
print(f"   Suicide data shape: {suicide_df.shape}")
print(f"   Years: {sorted(suicide_df['year'].unique())}")

# Unemployment data
print("2. Processing merged_unemployment_data.csv...")
unemployment_df = process_unemployment_data('merged_unemployment_data.csv')
print(f"   Unemployment data shape: {unemployment_df.shape}")
print(f"   Years: {sorted(unemployment_df['year'].unique())}")

# Urban population data
print("3. Processing merged_urban_population_data.csv...")
urban_df = process_urban_population_data('merged_urban_population_data.csv')
print(f"   Urban population data shape: {urban_df.shape}")
print(f"   Years: {sorted(urban_df['year'].unique())}")

# Merge all three datasets step by step
print("\nMerging all three datasets...")

# Start with suicide data as base
master_df = suicide_df
print(f"Starting with Suicide data as base: {master_df.shape}")

# Merge with Unemployment data
master_df = pd.merge(master_df, unemployment_df, on=['country', 'year'], how='outer')
print(f"After merging Unemployment data: {master_df.shape}")

# Merge with Urban Population data
master_df = pd.merge(master_df, urban_df, on=['country', 'year'], how='outer')
print(f"After merging Urban Population data: {master_df.shape}")

# Data quality checks
print("\n📊 DATA QUALITY REPORT:")
print(f"Total records: {len(master_df)}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Years covered: {sorted(master_df['year'].dropna().unique())}")

print("\nMissing values by column:")
missing_data = master_df.isnull().sum()
missing_pct = (master_df.isnull().sum() / len(master_df)) * 100
missing_report = pd.DataFrame({
    'Missing_Count': missing_data,
    'Missing_Percentage': missing_pct
})
print(missing_report.sort_values('Missing_Percentage', ascending=False).head(15))

# Calculate coverage statistics
print("\n📈 DATA COVERAGE STATISTICS:")
total_country_years = len(master_df)
coverage_stats = {}

for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct

coverage_df = pd.DataFrame({
    'Column': list(coverage_stats.keys()),
    'Coverage_Percentage': list(coverage_stats.values())
}).sort_values('Coverage_Percentage', ascending=False)

print(coverage_df.head(20))

# Save the merged dataset
output_file = 'merged/merged_suicide_unemployment_urban_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"\nMERGED DATASET SAVED AS: {output_file}")

# Final summary
print(f"\n🎯 MERGE COMPLETE!")
print(f"Total datasets merged: 3")
print(f"Final columns: {len(master_df.columns)}")
print("Key columns in final dataset:")
key_columns = ['suicide_rate', 'unemployment_rate', 'urban_population_pct', 'region', 'income_group']
for col in key_columns:
    if col in master_df.columns:
        non_missing = master_df[col].notna().sum()
        pct = (non_missing / len(master_df)) * 100
        print(f"  - {col}: {non_missing}/{len(master_df)} ({pct:.1f}%)")

print(f"\nTime range: 2011-2024")
print(f"Countries in final dataset: {master_df['country'].nunique()}")
print(f"Total country-year observations: {len(master_df)}")

# Show sample of the merged data
print("\n📋 SAMPLE OF MERGED DATA (first 8 rows):")
print(master_df.head(8))

# Show regions and income groups available
print(f"\n🌍 REGIONS IN DATASET:")
if 'region' in master_df.columns:
    regions = master_df['region'].dropna().unique()
    print(f"Regions: {list(regions)}")
    print(f"Number of countries with region data: {master_df['region'].notna().sum()}")

print(f"\n💰 INCOME GROUPS IN DATASET:")
if 'income_group' in master_df.columns:
    income_groups = master_df['income_group'].dropna().unique()
    print(f"Income groups: {list(income_groups)}")
    print(f"Number of countries with income group data: {master_df['income_group'].notna().sum()}")

# Show risk categories summary
print(f"\n⚠️  SUICIDE RISK CATEGORIES:")
risk_cols = ['suicide_very_high_risk', 'suicide_high_risk', 'suicide_moderate_risk', 'suicide_low_risk']
for col in risk_cols:
    if col in master_df.columns:
        count = master_df[col].sum()
        print(f"  - {col}: {count} countries")

print(f"\n💼 UNEMPLOYMENT CATEGORIES:")
unemp_cols = ['unemployment_very_high', 'unemployment_high', 'unemployment_moderate', 'unemployment_low']
for col in unemp_cols:
    if col in master_df.columns:
        count = master_df[col].sum()
        print(f"  - {col}: {count} countries")

print(f"\n🏙️  URBANIZATION CATEGORIES:")
urban_cols = ['highly_urbanized', 'moderately_urbanized', 'low_urbanized', 'very_low_urbanized', 'fully_urbanized']
for col in urban_cols:
    if col in master_df.columns:
        count = master_df[col].sum()
        print(f"  - {col}: {count} countries")

Processing the three datasets...
1. Processing merged_suicide_rate_data.csv...
   Suicide data shape: (2582, 10)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
2. Processing merged_unemployment_data.csv...
   Unemployment data shape: (3556, 12)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
3. Processing merged_urban_population_data.csv...
   Urban population data shape: (3542, 11)
   Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Merging all three datasets...
Starting with Suicide data as base: (2582, 10)
After merging Unemployment data: (3985, 20)
After merging Urban Population data: (3989, 29)

📊 DATA QUALITY REPORT:
Total records: 3989
Unique countries: 296
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values by column:
                                Missing_Count  Missing_Percentage
m

### Final Combinations

In [12]:
import pandas as pd
import numpy as np

def load_and_merge_datasets():
    """Load and merge the three specified datasets"""
    
    print("Loading the three datasets...")
    
    # Load dataset 1: comprehensive_merged_dataset_indexed.csv
    print("1. Loading comprehensive_merged_dataset_indexed.csv...")
    df1 = pd.read_csv('merged/comprehensive_merged_dataset_indexed.csv')
    # Drop the country_year column as we'll recreate it if needed
    if 'country_year' in df1.columns:
        df1 = df1.drop('country_year', axis=1)
    print(f"   Dataset 1 shape: {df1.shape}")
    
    # Load dataset 2: comprehensive_poverty_fertility_consumptionPerCapita_2011_2024_v2.csv
    print("2. Loading comprehensive_poverty_fertility_consumptionPerCapita_2011_2024_v2.csv...")
    df2 = pd.read_csv('merged/comprehensive_poverty_fertility_consumptionPerCapita_2011_2024_v2.csv')
    print(f"   Dataset 2 shape: {df2.shape}")
    
    # Load dataset 3: merged_alcohol_corruption_2011_2024.csv
    print("3. Loading merged_alcohol_corruption_2011_2024.csv...")
    df3 = pd.read_csv('merged/merged_alcohol_corruption_2011_2024.csv')
    print(f"   Dataset 3 shape: {df3.shape}")
    
    # Merge datasets step by step using outer joins
    print("\nMerging datasets...")
    
    # Start with dataset 1 as base
    master_df = df1
    print(f"Starting with Dataset 1 as base: {master_df.shape}")
    
    # Merge with dataset 2
    master_df = pd.merge(master_df, df2, on=['country', 'year'], how='outer', suffixes=('', '_dup'))
    print(f"After merging Dataset 2: {master_df.shape}")
    
    # Merge with dataset 3
    master_df = pd.merge(master_df, df3, on=['country', 'year'], how='outer')
    print(f"After merging Dataset 3: {master_df.shape}")
    
    # Handle duplicate columns (remove _dup suffix columns)
    duplicate_cols = [col for col in master_df.columns if col.endswith('_dup')]
    if duplicate_cols:
        print(f"Removing {len(duplicate_cols)} duplicate columns...")
        # Keep the original columns, drop the _dup ones
        original_cols = [col.replace('_dup', '') for col in duplicate_cols]
        for orig_col, dup_col in zip(original_cols, duplicate_cols):
            # If original column has missing values but duplicate has data, fill them
            mask = master_df[orig_col].isna() & master_df[dup_col].notna()
            master_df.loc[mask, orig_col] = master_df.loc[mask, dup_col]
        master_df = master_df.drop(duplicate_cols, axis=1)
        print(f"After removing duplicates: {master_df.shape}")
    
    return master_df

# Load and merge all datasets
print("🚀 MERGING ALL THREE DATASETS INTO ONE COMPREHENSIVE DATASET")
master_df = load_and_merge_datasets()

# Data quality checks and cleaning
print("\n📊 DATA QUALITY CHECKS:")

# Remove completely empty columns
empty_cols = []
for col in master_df.columns:
    if master_df[col].isna().all():
        empty_cols.append(col)

if empty_cols:
    print(f"Removing {len(empty_cols)} completely empty columns: {empty_cols}")
    master_df = master_df.drop(empty_cols, axis=1)

print(f"Final dataset shape: {master_df.shape}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Years covered: {sorted(master_df['year'].dropna().unique())}")

# Calculate coverage statistics
print("\n📈 DATA COVERAGE STATISTICS:")
total_country_years = len(master_df)
coverage_stats = {}

# Group columns by category for better organization
column_categories = {
    'Education': ['education_expenditure_pct_gdp'],
    'Happiness & Well-being': ['happiness_score', 'happiness_yearly_change', 'happiness_global_percentile', 
                             'very_high_happiness', 'very_low_happiness'],
    'Health & Obesity': ['overweight_pct', 'obesity_yearly_change', 'obesity_global_percentile',
                       'high_obesity', 'very_high_obesity'],
    'Poverty': ['poverty_rate', 'poverty_yearly_change', 'poverty_global_percentile',
               'extreme_poverty', 'moderate_poverty', 'low_poverty', 'zero_poverty'],
    'Demographics': ['fertility_rate', 'consumption_per_capita_usd'],
    'Substance Use': ['alcohol_consumption'],
    'Governance': ['corruption_index']
}

for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct

# Print coverage by category
for category, columns in column_categories.items():
    available_cols = [col for col in columns if col in master_df.columns]
    if available_cols:
        avg_coverage = np.mean([coverage_stats[col] for col in available_cols])
        print(f"{category}: {len(available_cols)} variables, avg coverage: {avg_coverage:.1f}%")

# Detailed coverage report
coverage_df = pd.DataFrame({
    'Column': list(coverage_stats.keys()),
    'Coverage_Percentage': list(coverage_stats.values())
}).sort_values('Coverage_Percentage', ascending=False)

print(f"\nTop 15 columns by coverage:")
print(coverage_df.head(15))

# Save the final comprehensive merged dataset
output_file = 'merged/FINAL_COMPREHENSIVE_MERGED_DATASET_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"\nFINAL COMPREHENSIVE MERGED DATASET SAVED AS: {output_file}")

# Final comprehensive summary
print(f"\n🎉 FINAL MERGE COMPLETE!")
print(f"==========================================")
print(f"📊 FINAL DATASET SUMMARY:")
print(f"==========================================")
print(f"Total datasets merged: 3")
print(f"Final columns: {len(master_df.columns)}")
print(f"Total country-year observations: {len(master_df)}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Time range: 2011-2024")
print(f"Total variables by category:")

for category, columns in column_categories.items():
    available_cols = [col for col in columns if col in master_df.columns]
    if available_cols:
        print(f"  - {category}: {len(available_cols)} variables")

print(f"\n🔍 SAMPLE OF FINAL DATA (first 5 rows):")
print(master_df.head())

print(f"\n📋 COLUMNS IN FINAL DATASET:")
for col in master_df.columns:
    non_missing = master_df[col].notna().sum()
    pct = (non_missing / len(master_df)) * 100
    print(f"  - {col}: {non_missing}/{len(master_df)} ({pct:.1f}%)")

print(f"\n🎯 READY FOR ANALYSIS!")
print(f"The comprehensive dataset contains data on:")
print(f"• Education expenditure")
print(f"• Happiness and well-being") 
print(f"• Health and obesity rates")
print(f"• Poverty metrics")
print(f"• Demographic indicators (fertility, consumption)")
print(f"• Alcohol consumption")
print(f"• Corruption perception")
print(f"• And much more!")

🚀 MERGING ALL THREE DATASETS INTO ONE COMPREHENSIVE DATASET
Loading the three datasets...
1. Loading comprehensive_merged_dataset_indexed.csv...
   Dataset 1 shape: (3592, 13)
2. Loading comprehensive_poverty_fertility_consumptionPerCapita_2011_2024_v2.csv...
   Dataset 2 shape: (4090, 22)
3. Loading merged_alcohol_corruption_2011_2024.csv...
   Dataset 3 shape: (2798, 4)

Merging datasets...
Starting with Dataset 1 as base: (3592, 13)
After merging Dataset 2: (4090, 33)
After merging Dataset 3: (4335, 35)
Removing 11 duplicate columns...
After removing duplicates: (4335, 24)

📊 DATA QUALITY CHECKS:
Final dataset shape: (4335, 24)
Unique countries: 322
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

📈 DATA COVERAGE STATISTICS:
Education: 1 variables, avg coverage: 58.7%
Happiness & Well-being: 5 variables, avg coverage: 44.9%
Health & Obesity: 5 variables, avg coverage: 55.4%
Poverty: 7 variables, avg coverage: 28.1%
Demographics: 2 

In [13]:
import pandas as pd
import numpy as np

def load_and_merge_final_datasets():
    """Load and merge the three final specified datasets"""
    
    print("Loading the three final datasets...")
    
    # Load dataset 1: GDP, Health, Homicide
    print("1. Loading merged_gdp_health_homicide_2011_2024.csv...")
    df1 = pd.read_csv('merged/merged_gdp_health_homicide_2011_2024.csv')
    print(f"   Dataset 1 shape: {df1.shape}")
    
    # Load dataset 2: Inflation, Internet, Population
    print("2. Loading merged_inflation_internet_population_2011_2024.csv...")
    df2 = pd.read_csv('merged/merged_inflation_internet_population_2011_2024.csv')
    print(f"   Dataset 2 shape: {df2.shape}")
    
    # Load dataset 3: Suicide, Unemployment, Urban
    print("3. Loading merged_suicide_unemployment_urban_2011_2024.csv...")
    df3 = pd.read_csv('merged/merged_suicide_unemployment_urban_2011_2024.csv')
    print(f"   Dataset 3 shape: {df3.shape}")
    
    # Merge datasets step by step using outer joins
    print("\nMerging final datasets...")
    
    # Start with dataset 1 as base
    master_df = df1
    print(f"Starting with GDP/Health/Homicide data as base: {master_df.shape}")
    
    # Merge with dataset 2
    master_df = pd.merge(master_df, df2, on=['country', 'year'], how='outer', suffixes=('', '_dup'))
    print(f"After merging Inflation/Internet/Population: {master_df.shape}")
    
    # Merge with dataset 3
    master_df = pd.merge(master_df, df3, on=['country', 'year'], how='outer', suffixes=('', '_dup2'))
    print(f"After merging Suicide/Unemployment/Urban: {master_df.shape}")
    
    # Handle duplicate columns
    print("\nHandling duplicate columns...")
    
    # Remove _dup and _dup2 suffix columns, keeping the originals
    duplicate_cols = [col for col in master_df.columns if col.endswith('_dup') or col.endswith('_dup2')]
    if duplicate_cols:
        print(f"Found {len(duplicate_cols)} duplicate columns to handle")
        
        # For each duplicate, check if we should merge data with original
        for dup_col in duplicate_cols:
            if dup_col.endswith('_dup'):
                orig_col = dup_col.replace('_dup', '')
            else:  # _dup2
                orig_col = dup_col.replace('_dup2', '')
            
            if orig_col in master_df.columns:
                # Fill missing values in original with data from duplicate
                mask = master_df[orig_col].isna() & master_df[dup_col].notna()
                master_df.loc[mask, orig_col] = master_df.loc[mask, dup_col]
        
        # Remove all duplicate columns
        master_df = master_df.drop(duplicate_cols, axis=1)
        print(f"After removing duplicates: {master_df.shape}")
    
    return master_df

# Load and merge all final datasets
print("🚀 MERGING FINAL THREE DATASETS INTO ULTIMATE COMPREHENSIVE DATASET")
master_df = load_and_merge_final_datasets()
# Data quality checks and cleaning
print("\n📊 DATA QUALITY CHECKS:")
# Remove completely empty columns
empty_cols = []
for col in master_df.columns:
    if master_df[col].isna().all():
        empty_cols.append(col)
if empty_cols:
    print(f"Removing {len(empty_cols)} completely empty columns: {empty_cols}")
    master_df = master_df.drop(empty_cols, axis=1)
print(f"Final dataset shape: {master_df.shape}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Years covered: {sorted(master_df['year'].dropna().unique())}")

# Calculate coverage statistics
print("\n📈 DATA COVERAGE STATISTICS:")
total_country_years = len(master_df)
coverage_stats = {}
# Group columns by category for better organization
column_categories = {
    'Economic Indicators': ['gdp_per_capita_ppp', 'inflation_pct', 'inflation_3yr_avg', 'unemployment_rate'],
    'Health & Safety': ['life_satisfaction', 'health_expenditure_per_capita_usd', 'homicide_rate_per_100k', 
                      'suicide_rate', 'male_suicide_rate', 'female_suicide_rate'],
    'Demographics': ['population', 'population_millions', 'population_growth_rate', 'urban_population_pct',
                   'urban_growth_rate', 'fertility_rate'],
    'Technology & Development': ['internet_users_pct'],
    'Social Indicators': ['suicide_global_percentile', 'unemployment_global_percentile', 'urban_global_percentile'],
    'Geographic': ['region', 'income_group'],
    'Risk Categories': ['suicide_very_high_risk', 'suicide_high_risk', 'suicide_moderate_risk', 'suicide_low_risk',
                      'unemployment_very_high', 'unemployment_high', 'unemployment_moderate', 'unemployment_low',
                      'highly_urbanized', 'moderately_urbanized', 'low_urbanized', 'very_low_urbanized', 'fully_urbanized',
                      'inflation_category', 'inflation_yoy_increase']
}
for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct
# Print coverage by category
print("\n📊 COVERAGE BY CATEGORY:")
for category, columns in column_categories.items():
    available_cols = [col for col in columns if col in master_df.columns]
    if available_cols:
        avg_coverage = np.mean([coverage_stats.get(col, 0) for col in available_cols])
        print(f"  {category}: {len(available_cols)} variables, avg coverage: {avg_coverage:.1f}%")

# Detailed coverage report
coverage_df = pd.DataFrame({
    'Column': list(coverage_stats.keys()),
    'Coverage_Percentage': list(coverage_stats.values())
}).sort_values('Coverage_Percentage', ascending=False)

print(f"\nTop 20 columns by coverage:")
print(coverage_df.head(20))

# Save the ultimate comprehensive merged dataset
output_file = 'merged/ULTIMATE_COMPREHENSIVE_DATASET_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"\n✅ ULTIMATE COMPREHENSIVE DATASET SAVED AS: {output_file}")

# Final comprehensive summary
print(f"\n🎉 ULTIMATE MERGE COMPLETE!")
print(f"==========================================")
print(f"📊 ULTIMATE DATASET SUMMARY:")
print(f"==========================================")
print(f"Total final datasets merged: 3")
print(f"Final columns: {len(master_df.columns)}")
print(f"Total country-year observations: {len(master_df)}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Time range: 2011-2024")
print(f"\nData categories included:")
for category, columns in column_categories.items():
    available_cols = [col for col in columns if col in master_df.columns]
    if available_cols:
        print(f"  • {category}: {len(available_cols)} variables")

print(f"\n🔍 SAMPLE OF ULTIMATE DATA (first 3 rows):")
print(master_df.head(3))

print(f"\n🌍 GEOGRAPHIC COVERAGE:")
if 'region' in master_df.columns:
    regions = master_df['region'].dropna().unique()
    print(f"Regions: {list(regions)}")
    print(f"Countries with region data: {master_df['region'].notna().sum()}")

if 'income_group' in master_df.columns:
    income_groups = master_df['income_group'].dropna().unique()
    print(f"Income groups: {list(income_groups)}")
    print(f"Countries with income group data: {master_df['income_group'].notna().sum()}")

print(f"\n📈 KEY INDICATORS SUMMARY:")
key_indicators = {
    'GDP per capita (PPP)': 'gdp_per_capita_ppp',
    'Life Satisfaction': 'life_satisfaction', 
    'Inflation Rate': 'inflation_pct',
    'Unemployment Rate': 'unemployment_rate',
    'Health Expenditure per capita': 'health_expenditure_per_capita_usd',
    'Internet Users %': 'internet_users_pct',
    'Urban Population %': 'urban_population_pct'
}

for name, col in key_indicators.items():
    if col in master_df.columns:
        non_missing = master_df[col].notna().sum()
        pct = (non_missing / len(master_df)) * 100
        if non_missing > 0:
            avg_val = master_df[col].mean()
            print(f"  {name}: {non_missing} records ({pct:.1f}%), avg: {avg_val:.2f}")

print(f"\n🎯 READY FOR ADVANCED ANALYSIS!")
print(f"The ultimate comprehensive dataset contains:")
print(f"• Economic indicators (GDP, inflation, unemployment)")
print(f"• Health and safety metrics") 
print(f"• Demographic and population data")
print(f"• Technology adoption (internet usage)")
print(f"• Social well-being indicators")
print(f"• Geographic and income classifications")
print(f"• Risk categories across multiple dimensions")
print(f"• Time-series data from 2011-2024")

🚀 MERGING FINAL THREE DATASETS INTO ULTIMATE COMPREHENSIVE DATASET
Loading the three final datasets...
1. Loading merged_gdp_health_homicide_2011_2024.csv...
   Dataset 1 shape: (4459, 7)
2. Loading merged_inflation_internet_population_2011_2024.csv...
   Dataset 2 shape: (3936, 13)
3. Loading merged_suicide_unemployment_urban_2011_2024.csv...
   Dataset 3 shape: (3989, 29)

Merging final datasets...
Starting with GDP/Health/Homicide data as base: (4459, 7)
After merging Inflation/Internet/Population: (4461, 18)
After merging Suicide/Unemployment/Urban: (4551, 45)

Handling duplicate columns...
Found 3 duplicate columns to handle
After removing duplicates: (4551, 42)

📊 DATA QUALITY CHECKS:
Final dataset shape: (4551, 42)
Unique countries: 342
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

📈 DATA COVERAGE STATISTICS:

📊 COVERAGE BY CATEGORY:
  Economic Indicators: 4 variables, avg coverage: 60.4%
  Health & Safety: 6 variables, avg 

In [15]:
import pandas as pd
import numpy as np

def create_final_master_dataset():
    """Merge the two final comprehensive datasets into one ultimate master dataset"""
    
    print("🚀 CREATING THE ULTIMATE MASTER DATASET")
    print("Merging the two final comprehensive datasets...")
    
    # Load Dataset 1: FINAL_COMPREHENSIVE_MERGED_DATASET_2011_2024.csv
    print("1. Loading FINAL_COMPREHENSIVE_MERGED_DATASET_2011_2024.csv...")
    df1 = pd.read_csv('merged/FINAL_COMPREHENSIVE_MERGED_DATASET_2011_2024.csv')
    print(f"   Dataset 1 shape: {df1.shape}")
    print(f"   Dataset 1 columns: {len(df1.columns)}")
    
    # Load Dataset 2: ULTIMATE_COMPREHENSIVE_DATASET_2011_2024.csv
    print("2. Loading ULTIMATE_COMPREHENSIVE_DATASET_2011_2024.csv...")
    df2 = pd.read_csv('merged/ULTIMATE_COMPREHENSIVE_DATASET_2011_2024.csv')
    print(f"   Dataset 2 shape: {df2.shape}")
    print(f"   Dataset 2 columns: {len(df2.columns)}")
    
    # Merge the two datasets
    print("\n🔄 MERGING DATASETS...")
    master_df = pd.merge(df1, df2, on=['country', 'year'], how='outer')
    print(f"   After merge: {master_df.shape}")
    
    # Handle duplicate columns (remove _x and _y suffixes if any)
    duplicate_cols = [col for col in master_df.columns if col.endswith('_x') or col.endswith('_y')]
    if duplicate_cols:
        print(f"   Handling {len(duplicate_cols)} potential duplicate columns...")
        # For now, we'll keep both and handle manually if needed
        # In practice, these should be the same data, so we can merge them
    
    return master_df

# Create the ultimate master dataset
print("=" * 70)
print("🎯 FINAL STAGE: CREATING ULTIMATE MASTER DATASET")
print("=" * 70)

master_df = create_final_master_dataset()

# Data quality and cleaning
print("\n📊 DATA QUALITY CHECKS:")

# Remove completely empty columns
empty_cols = []
for col in master_df.columns:
    if master_df[col].isna().all():
        empty_cols.append(col)

if empty_cols:
    print(f"Removing {len(empty_cols)} completely empty columns")
    master_df = master_df.drop(empty_cols, axis=1)

print(f"Final dataset shape: {master_df.shape}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Years covered: {sorted(master_df['year'].dropna().unique())}")

# Calculate comprehensive coverage statistics
print("\n📈 COMPREHENSIVE DATA COVERAGE ANALYSIS:")

total_country_years = len(master_df)
coverage_stats = {}

# Define comprehensive categories
comprehensive_categories = {
    'Education & Development': [
        'education_expenditure_pct_gdp', 'internet_users_pct'
    ],
    'Well-being & Happiness': [
        'happiness_score', 'happiness_yearly_change', 'happiness_global_percentile',
        'very_high_happiness', 'very_low_happiness', 'life_satisfaction'
    ],
    'Health & Demographics': [
        'overweight_pct', 'obesity_yearly_change', 'obesity_global_percentile',
        'high_obesity', 'very_high_obesity', 'fertility_rate', 'population',
        'population_millions', 'population_growth_rate', 'urban_population_pct',
        'urban_growth_rate', 'health_expenditure_per_capita_usd'
    ],
    'Economic Indicators': [
        'gdp_per_capita_ppp', 'consumption_per_capita_usd', 'inflation_pct',
        'inflation_3yr_avg', 'unemployment_rate', 'unemployment_growth_rate'
    ],
    'Social Issues': [
        'poverty_rate', 'poverty_global_percentile', 'extreme_poverty',
        'moderate_poverty', 'low_poverty', 'zero_poverty', 'suicide_rate',
        'male_suicide_rate', 'female_suicide_rate', 'homicide_rate_per_100k'
    ],
    'Substance & Governance': [
        'alcohol_consumption', 'corruption_index'
    ],
    'Risk Categories': [
        'suicide_very_high_risk', 'suicide_high_risk', 'suicide_moderate_risk', 'suicide_low_risk',
        'unemployment_very_high', 'unemployment_high', 'unemployment_moderate', 'unemployment_low',
        'highly_urbanized', 'moderately_urbanized', 'low_urbanized', 'very_low_urbanized', 'fully_urbanized',
        'inflation_category', 'inflation_yoy_increase'
    ],
    'Geographic & Classification': [
        'region', 'income_group'
    ]
}

for col in master_df.columns:
    if col not in ['country', 'year']:
        non_missing = master_df[col].notna().sum()
        coverage_pct = (non_missing / total_country_years) * 100
        coverage_stats[col] = coverage_pct

# Print coverage by comprehensive categories
print("\n📊 COVERAGE BY COMPREHENSIVE CATEGORIES:")
total_variables = 0
for category, columns in comprehensive_categories.items():
    available_cols = [col for col in columns if col in master_df.columns]
    if available_cols:
        avg_coverage = np.mean([coverage_stats.get(col, 0) for col in available_cols])
        total_variables += len(available_cols)
        print(f"  {category}: {len(available_cols)} variables, avg coverage: {avg_coverage:.1f}%")

print(f"\n📦 TOTAL VARIABLES IN MASTER DATASET: {total_variables}")

# Save the ultimate master dataset
print(f"\n💾 SAVING ULTIMATE MASTER DATASET...")

# Main master file
output_file = 'ULTIMATE_MASTER_DATASET_2011_2024.csv'
master_df.to_csv(output_file, index=False)
print(f"✅ ULTIMATE MASTER DATASET SAVED AS: {output_file}")

# Final comprehensive summary
print(f"\n🎉 ULTIMATE MASTER DATASET CREATION COMPLETE!")
print("=" * 70)
print(f"📊 FINAL SUMMARY:")
print("=" * 70)
print(f"Total datasets merged in final stage: 2")
print(f"Final columns in master dataset: {len(master_df.columns)}")
print(f"Total country-year observations: {len(master_df)}")
print(f"Unique countries: {master_df['country'].nunique()}")
print(f"Time range: 2011-2024")
print(f"Total variables across all categories: {total_variables}")

print(f"\n🌍 GEOGRAPHIC COVERAGE:")
if 'region' in master_df.columns:
    regions = master_df['region'].dropna().unique()
    print(f"Regions covered: {len(regions)}")
    print(f"Countries with region data: {master_df['region'].notna().sum()}")

print(f"\n💰 INCOME GROUP DISTRIBUTION:")
if 'income_group' in master_df.columns:
    income_counts = master_df['income_group'].value_counts()
    for group, count in income_counts.items():
        print(f"  {group}: {count} country-years")

print(f"\n📈 KEY INDICATOR AVAILABILITY:")
key_indicators = {
    'GDP per capita': 'gdp_per_capita_ppp',
    'Life Satisfaction': 'life_satisfaction',
    'Happiness Score': 'happiness_score', 
    'Education Expenditure': 'education_expenditure_pct_gdp',
    'Health Expenditure': 'health_expenditure_per_capita_usd',
    'Inflation Rate': 'inflation_pct',
    'Unemployment Rate': 'unemployment_rate',
    'Internet Users %': 'internet_users_pct',
    'Urban Population %': 'urban_population_pct',
    'Fertility Rate': 'fertility_rate',
    'Corruption Index': 'corruption_index'
}

for name, col in key_indicators.items():
    if col in master_df.columns:
        non_missing = master_df[col].notna().sum()
        pct = (non_missing / len(master_df)) * 100
        print(f"  {name}: {non_missing} records ({pct:.1f}%)")

print(f"\n🔬 DATASET READY FOR:")
print(f"• Cross-country comparative analysis")
print(f"• Time-series studies (2011-2024)")
print(f"• Multidimensional research")
print(f"• Economic development studies") 
print(f"• Social well-being analysis")
print(f"• Health and demographic research")
print(f"• Policy impact evaluation")

print(f"\n🎯 ANALYSIS POTENTIAL:")
print(f"This dataset enables research across:")
print(f"• Economic development vs. well-being")
print(f"• Health outcomes vs. economic factors")
print(f"• Education investment impacts")
print(f"• Urbanization trends")
print(f"• Governance and corruption effects")
print(f"• And much more!")

print(f"\n⭐ ULTIMATE MASTER DATASET COMPLETE! ⭐")

🎯 FINAL STAGE: CREATING ULTIMATE MASTER DATASET
🚀 CREATING THE ULTIMATE MASTER DATASET
Merging the two final comprehensive datasets...
1. Loading FINAL_COMPREHENSIVE_MERGED_DATASET_2011_2024.csv...
   Dataset 1 shape: (4335, 24)
   Dataset 1 columns: 24
2. Loading ULTIMATE_COMPREHENSIVE_DATASET_2011_2024.csv...
   Dataset 2 shape: (4551, 42)
   Dataset 2 columns: 42

🔄 MERGING DATASETS...
   After merge: (4972, 64)

📊 DATA QUALITY CHECKS:
Final dataset shape: (4972, 64)
Unique countries: 374
Years covered: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

📈 COMPREHENSIVE DATA COVERAGE ANALYSIS:

📊 COVERAGE BY COMPREHENSIVE CATEGORIES:
  Education & Development: 2 variables, avg coverage: 54.4%
  Well-being & Happiness: 6 variables, avg coverage: 39.3%
  Health & Demographics: 12 variables, avg coverage: 57.6%
  Economic Indicators: 6 variables, avg coverage: 54.0%
  Social Issues: 10 variables, avg coverage: 25.3%
  Substance & Governance: 2 variable

## Pre-Processing on the Final Dataset

### Feature Preservaton & Normalisation

In [17]:
import pandas as pd

# Load dataset
df = pd.read_csv("HappinesssGlobalData_v1.csv")

# -----------------------------
# 1. Normalize country names
# -----------------------------
def normalize_country(name):
    if pd.isna(name):
        return name
    name = name.strip()
    name = name.title()

    replacements = {
        "Usa": "United States",
        "Uae": "United Arab Emirates",
        "Uk": "United Kingdom",
        "Bolivia (Plurinational State Of)": "Bolivia",
        "Venezuela (Bolivarian Republic Of)": "Venezuela",
        "Iran, Islamic Republic Of": "Iran",
        "Korea, Republic Of": "South Korea",
        "Korea, Democratic People's Republic Of": "North Korea",
    }
    
    return replacements.get(name, name)

df["country"] = df["country"].apply(normalize_country)


# -----------------------------
# 2. Convert True/False columns to real booleans (or 0/1 if float)
# -----------------------------
bool_cols = [
    col for col in df.columns
    if df[col].dtype == object and df[col].dropna().isin(["True", "False"]).all()
]

for col in bool_cols:
    df[col] = df[col].map({"True": True, "False": False})


# -----------------------------
# 3. Convert numeric-like object columns to numeric
# -----------------------------
for col in df.columns:
    if df[col].dtype == object:
        df[col] = pd.to_numeric(df[col], errors="ignore")


# -----------------------------
# 4. Strip whitespace in string columns
# -----------------------------
for col in df.select_dtypes(include="object"):
    df[col] = df[col].astype(str).str.strip()


# -----------------------------
# DONE. Cleaned dataframe is in df
# -----------------------------

print("Shape:", df.shape)
print(df.head())
df.info()

# Save cleaned dataset
df_cleaned.to_csv("HappinesssGlobalData_v3.csv", index=False)

C:\Users\ibrahim\AppData\Local\Temp\ipykernel_9340\1400771587.py:48: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors="ignore")


Shape: (4972, 64)
       country  year  education_expenditure_pct_gdp  happiness_score  \
0  Afghanistan  2011                        3.46201            4.258   
1  Afghanistan  2012                        2.60420            4.040   
2  Afghanistan  2013                        3.45446              NaN   
3  Afghanistan  2014                        3.69522            3.575   
4  Afghanistan  2015                        3.25580            3.360   

   happiness_yearly_change  happiness_global_percentile  very_high_happiness  \
0                      NaN                    16.167665                  0.0   
1                   -0.218                     8.383234                  0.0   
2                      NaN                          NaN                  NaN   
3                   -0.465                     3.550296                  0.0   
4                   -0.215                     2.380952                  0.0   

   very_low_happiness  overweight_pct  obesity_yearly_change  ...  \

### Standardization of Country Names, and Merging

In [5]:
import pandas as pd
import numpy as np

# Read the CSV file
df = pd.read_csv('HappinesssGlobalData_v1.csv')

# Define country name mappings for consolidation
country_mappings = {
    # Regional/Aggregate entries to remove
    'Arab World': None,
    'Caribbean small states': None,
    'Central Europe and the Baltics': None,
    'Early-demographic dividend': None,
    'East Asia & Pacific': None,
    'East Asia & Pacific (IDA & IBRD countries)': None,
    'East Asia & Pacific (excluding high income)': None,
    'East Asia and Pacific (WB)': None,
    'Euro area': None,
    'Europe': None,
    'Europe & Central Asia': None,
    'Europe & Central Asia (IDA & IBRD countries)': None,
    'Europe & Central Asia (excluding high income)': None,
    'Europe and Central Asia (WB)': None,
    'European Union': None,
    'Fragile and conflict affected situations': None,
    'Heavily indebted poor countries (HIPC)': None,
    'High income': None,
    'IBRD only': None,
    'IDA & IBRD total': None,
    'IDA blend': None,
    'IDA only': None,
    'IDA total': None,
    'Late-demographic dividend': None,
    'Latin America & Caribbean': None,
    'Latin America & Caribbean (excluding high income)': None,
    'Latin America & the Caribbean (IDA & IBRD countries)': None,
    'Latin America and Caribbean (WB)': None,
    'Least developed countries: UN classification': None,
    'Low & middle income': None,
    'Low income': None,
    'Lower middle income': None,
    'Middle East & North Africa': None,
    'Middle East, North Africa, Afghanistan and Pakistan (WB)': None,
    'Middle income': None,
    'North America': None,
    'North America (WB)': None,
    'Not classified': None,
    'OECD members': None,
    'Oceania': None,
    'Other small states': None,
    'Pacific island small states': None,
    'Post-demographic dividend': None,
    'Pre-demographic dividend': None,
    'Small states': None,
    'South Asia': None,
    'South Asia (IDA & IBRD)': None,
    'South Asia (WB)': None,
    'Sub-Saharan Africa (WB)': None,
    'Upper middle income': None,
    'World': None,
    'World (excluding China)': None,
    'World (excluding India)': None,
    'Eastern and Southern Africa (WB)': None,
    'Western and Central Africa (WB)': None,
    'Asia': None,
    
    # Country name standardizations
    'Argentina (urban)': 'Argentina',
    'Bahamas, The': 'Bahamas',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Brunei Darussalam': 'Brunei',
    'Cabo Verde': 'Cape Verde',
    'China (rural)': 'China',
    'China (urban)': 'China',
    'Congo, Dem. Rep.': 'Democratic Republic of Congo',
    'Congo, Rep.': 'Republic of the Congo',
    'Côte d\'Ivoire': 'Ivory Coast',
    'CÃ´te d\'Ivoire': 'Ivory Coast',
    'Cote d\'Ivoire': 'Ivory Coast',
    'Czechia': 'Czech Republic',
    'Egypt, Arab Rep.': 'Egypt',
    'Gambia, The': 'Gambia',
    'Hong Kong SAR, China': 'Hong Kong',
    'Iran, Islamic Rep.': 'Iran',
    'Korea, Rep.': 'South Korea',
    'Korea, Dem. People\'s Rep.': 'North Korea',
    'Korea, North': 'North Korea',
    'Korea, South': 'South Korea',
    'Kyrgyz Republic': 'Kyrgyzstan',
    'Lao PDR': 'Laos',
    'Macao SAR, China': 'Macao',
    'Micronesia, Fed. Sts.': 'Micronesia (country)',
    'Puerto Rico (US)': 'Puerto Rico',
    'St. Kitts and Nevis': 'Saint Kitts and Nevis',
    'St. Lucia': 'Saint Lucia',
    'St. Martin (French part)': 'Saint Martin (French part)',
    'St. Vincent and the Grenadines': 'Saint Vincent and the Grenadines',
    'Slovak Republic': 'Slovakia',
    'Turkiye': 'Turkey',
    'TÃ¼rkiye': 'Turkey',
    'United States of America': 'United States',
    'Venezuela, RB': 'Venezuela',
    'Viet Nam': 'Vietnam',
    'Virgin Islands (U.S.)': 'United States Virgin Islands',
    'West Bank and Gaza': 'Palestine',
    'Yemen, Rep.': 'Yemen',
    
    # Historical entities to remove
    'Austria-Hungary': None,
    'Czechoslovakia': None,
    'East Germany': None,
    'West Germany': None,
    'Korea (former)': None,
    'Serbia and Montenegro': None,
    'USSR': None,
    'Yemen Arab Republic': None,
    'Yemen People\'s Republic': None,
    'Yugoslavia': None,
    'Ethiopia (former)': None,
    'Republic of Vietnam': None,
    
    # Duplicate entries
    'Democratic People\'s Republic of Korea': 'North Korea',
    'Republic of Korea': 'South Korea',
    'Republic of Moldova': 'Moldova',
    'Lao People\'s Democratic Republic': 'Laos',
    'United Republic of Tanzania': 'Tanzania',
    'Syrian Arab Republic': 'Syria',
    'United Kingdom of Great Britain and Northern Ireland': 'United Kingdom',
    'Venezuela (Bolivarian Republic of)': 'Venezuela',
    'Bolivia (Plurinational State of)': 'Bolivia',
    'Iran (Islamic Republic of)': 'Iran',
    'Micronesia (Federated States of)': 'Micronesia (country)',
    'Somalia, Fed. Rep.': 'Somalia',
    'DR Congo': 'Democratic Republic of Congo',
    'Republic of the Congo': 'Congo',
    'Netherlands (Kingdom of the)': 'Netherlands',
    
    # Empty/invalid entries
    'Indicator Name': None,
    'Current health expenditure per capita (current US$)': None,
    '': None
}

# Apply country name mappings
df['country'] = df['country'].replace(country_mappings)

# Remove rows where country is None (regional aggregates and invalid entries)
df = df[df['country'].notna()]

# Remove completely empty rows
df = df.dropna(how='all')

# Group by country and year, then merge
def merge_records(group):
    """Merge records with priority for non-null and higher values"""
    if len(group) == 1:
        return group.iloc[0]
    
    result = {}
    for col in group.columns:
        values = group[col].dropna()
        
        if len(values) == 0:
            result[col] = np.nan
        elif col in ['country', 'year']:
            # Keep the first non-null value for identifiers
            result[col] = values.iloc[0]
        else:
            # For numeric columns, take the maximum non-null value
            try:
                result[col] = values.max()
            except:
                result[col] = values.iloc[0]
    
    return pd.Series(result)

# Group by country and year, then merge duplicate records
df_cleaned = df.groupby(['country', 'year'], as_index=False).apply(merge_records)

# Reset index
df_cleaned = df_cleaned.reset_index(drop=True)

# Sort by country and year
df_cleaned = df_cleaned.sort_values(['country', 'year']).reset_index(drop=True)

# Save the cleaned dataset
df_cleaned.to_csv('HappinesssGlobalData_v4.csv', index=False)

# Print summary statistics
print(f"Original dataset: {len(df)} rows")
print(f"Cleaned dataset: {len(df_cleaned)} rows")
print(f"Unique countries: {df_cleaned['country'].nunique()}")
print(f"\nCountries in cleaned dataset:")
print(df_cleaned['country'].value_counts().sort_index())
print(f"\nCleaned dataset saved as 'HappinesssGlobalData_v4.csv'")

C:\Users\ibrahim\AppData\Local\Temp\ipykernel_7608\1106082599.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_cleaned = df.groupby(['country', 'year'], as_index=False).apply(merge_records)


Original dataset: 3821 rows
Cleaned dataset: 3221 rows
Unique countries: 256

Countries in cleaned dataset:
country
Afghanistan          14
Aland Islands         1
Albania              14
Algeria              14
American Samoa       14
                     ..
Wallis and Futuna     1
Western Sahara        1
Yemen                14
Zambia               14
Zimbabwe             14
Name: count, Length: 256, dtype: int64

Cleaned dataset saved as 'HappinesssGlobalData_v4.csv'


In [7]:
import pandas as pd
import numpy as np

def clean_country_dataset(df):
    """
    Clean the dataset by merging duplicate country entries and standardizing the data
    """
    
    # Create a copy to avoid modifying the original
    df_clean = df.copy()
    
    # First, let's identify and standardize country names
    def standardize_country_name(name):
        """Standardize country names by removing special annotations"""
        if pd.isna(name):
            return name
        
        # Remove common annotations in parentheses
        name = str(name).split('(')[0].strip()
        
        # Remove "The" from beginning
        if name.startswith('"'):
            name = name.replace('"', '')
        if name.startswith('The '):
            name = name[4:]
            
        # Specific known duplicates to handle
        replacements = {
            'Bahamas, The': 'Bahamas',
            'Congo, Dem. Rep.': 'Democratic Republic of Congo',
            'Congo, Rep.': 'Republic of Congo',
            'Egypt, Arab Rep.': 'Egypt',
            'Brunei Darussalam': 'Brunei',
            'Cabo Verde': 'Cape Verde',
            'Cote d\'Ivoire': 'Ivory Coast',
            'Côte d\'Ivoire': 'Ivory Coast',
            'Eswatini': 'Swaziland',  # Eswatini is the new name for Swaziland
        }
        
        return replacements.get(name, name)
    
    # Apply country name standardization
    df_clean['country'] = df_clean['country'].apply(standardize_country_name)
    
    # Remove regional/economic groupings (keep only actual countries)
    non_country_entities = [
        'Arab World', 'Asia', 'Euro area', 'Europe', 'European Union',
        'Caribbean small states', 'Central Europe and the Baltics',
        'Early-demographic dividend', 'East Asia & Pacific',
        'East Asia & Pacific (IDA & IBRD countries)',
        'East Asia & Pacific (excluding high income)',
        'East Asia and Pacific (WB)', 'Eastern and Southern Africa (WB)',
        'Europe & Central Asia', 'Europe & Central Asia (IDA & IBRD countries)',
        'Europe & Central Asia (excluding high income)',
        'Europe and Central Asia (WB)', 'Fragile and conflict affected situations',
        'Central Europe and the Baltics', 'Current health expenditure per capita (current US$)',
        'Aland Islands', 'American Samoa', 'Anguilla', 'Bermuda', 'British Indian Ocean Territory',
        'British Virgin Islands', 'Cayman Islands', 'Channel Islands', 'Christmas Island',
        'Cocos Islands', 'Cook Islands', 'Curacao', 'Falkland Islands', 'Faroe Islands',
        'French Guiana', 'French Polynesia', 'Gibraltar', 'Greenland', 'Guam', 'Guernsey',
        'Isle of Man', 'Jersey', 'Macao', 'New Caledonia', 'Niue', 'Norfolk Island',
        'Northern Mariana Islands', 'Palau', 'Pitcairn Islands', 'Puerto Rico',
        'Reunion', 'Saint Barthelemy', 'Saint Helena', 'Saint Martin', 'Saint Pierre and Miquelon',
        'Samoa', 'Sint Maarten', 'Tokelau', 'Turks and Caicos Islands', 'Tuvalu',
        'US Virgin Islands', 'Wallis and Futuna', 'West Bank and Gaza'
    ]
    
    df_clean = df_clean[~df_clean['country'].isin(non_country_entities)]
    
    # List of recognized sovereign states (UN member states + observer states)
    recognized_countries = [
        'Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda',
        'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas',
        'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin',
        'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei',
        'Bulgaria', 'Burkina Faso', 'Burundi', 'Cape Verde', 'Cambodia', 'Cameroon',
        'Canada', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia',
        'Comoros', 'Democratic Republic of Congo', 'Republic of Congo', 'Costa Rica',
        'Ivory Coast', 'Croatia', 'Cuba', 'Cyprus', 'Czechia', 'Denmark', 'Djibouti',
        'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador',
        'Equatorial Guinea', 'Eritrea', 'Estonia', 'Swaziland', 'Ethiopia', 'Fiji',
        'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece',
        'Grenada', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras',
        'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel',
        'Italy', 'Jamaica', 'Japan', 'Jordan', 'Kazakhstan', 'Kenya', 'Kiribati',
        'Kuwait', 'Kyrgyzstan', 'Laos', 'Latvia', 'Lebanon', 'Lesotho', 'Liberia',
        'Libya', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Madagascar', 'Malawi',
        'Malaysia', 'Maldives', 'Mali', 'Malta', 'Marshall Islands', 'Mauritania',
        'Mauritius', 'Mexico', 'Micronesia', 'Moldova', 'Monaco', 'Mongolia',
        'Montenegro', 'Morocco', 'Mozambique', 'Myanmar', 'Namibia', 'Nauru', 'Nepal',
        'Netherlands', 'New Zealand', 'Nicaragua', 'Niger', 'Nigeria', 'North Korea',
        'North Macedonia', 'Norway', 'Oman', 'Pakistan', 'Palau', 'Panama',
        'Papua New Guinea', 'Paraguay', 'Peru', 'Philippines', 'Poland', 'Portugal',
        'Qatar', 'Romania', 'Russia', 'Rwanda', 'Saint Kitts and Nevis', 'Saint Lucia',
        'Saint Vincent and the Grenadines', 'Samoa', 'San Marino', 'Sao Tome and Principe',
        'Saudi Arabia', 'Senegal', 'Serbia', 'Seychelles', 'Sierra Leone', 'Singapore',
        'Slovakia', 'Slovenia', 'Solomon Islands', 'Somalia', 'South Africa', 'South Korea',
        'South Sudan', 'Spain', 'Sri Lanka', 'Sudan', 'Suriname', 'Sweden', 'Switzerland',
        'Syria', 'Taiwan', 'Tajikistan', 'Tanzania', 'Thailand', 'Timor-Leste', 'Togo',
        'Tonga', 'Trinidad and Tobago', 'Tunisia', 'Turkey', 'Turkmenistan', 'Tuvalu',
        'Uganda', 'Ukraine', 'United Arab Emirates', 'United Kingdom', 'United States',
        'Uruguay', 'Uzbekistan', 'Vanuatu', 'Vatican City', 'Venezuela', 'Vietnam',
        'Yemen', 'Zambia', 'Zimbabwe'
    ]
    
    df_clean = df_clean[df_clean['country'].isin(recognized_countries)]
    
    # Function to merge rows for the same country and year
    def merge_country_rows(group):
        """Merge multiple rows for the same country and year"""
        if len(group) == 1:
            return group.iloc[0]
        
        # Start with the first row
        merged_row = group.iloc[0].copy()
        
        # For each column, apply merging rules
        for col in group.columns:
            if col in ['country', 'year']:
                continue
                
            values = group[col].dropna()
            
            if len(values) == 0:
                # All values are NaN
                merged_row[col] = np.nan
            elif col in ['population', 'population_millions', 'urban_population_pct', 
                        'urban_global_percentile']:
                # For numerical columns, take the maximum value
                merged_row[col] = values.max()
            elif col in ['unemployment_very_high', 'unemployment_high', 
                        'unemployment_moderate', 'unemployment_low',
                        'highly_urbanized', 'moderately_urbanized', 
                        'low_urbanized', 'very_low_urbanized', 'fully_urbanized']:
                # For boolean columns, use OR logic (if any is True, result is True)
                merged_row[col] = values.any() if len(values) > 0 else False
            else:
                # For other columns, take the first non-null value
                merged_row[col] = values.iloc[0] if len(values) > 0 else np.nan
        
        return merged_row
    
    # Group by country and year, then merge
    df_clean = df_clean.groupby(['country', 'year']).apply(merge_country_rows).reset_index(drop=True)
    
    # Final cleanup: remove any remaining duplicates
    df_clean = df_clean.drop_duplicates(subset=['country', 'year'])
    
    return df_clean

# Load your dataset
df = pd.read_csv('HappinesssGlobalData_v4.csv')

# Apply the cleaning function
df_cleaned = clean_country_dataset(df)

# Print summary statistics
print(f"Original number of records: {len(df)}")
print(f"Cleaned number of records: {len(df_cleaned)}")
print(f"Number of unique countries: {df_cleaned['country'].nunique()}")
print(f"Number of unique years: {df_cleaned['year'].nunique()}")

# Show the unique countries in the cleaned dataset
print("\nUnique countries in cleaned dataset:")
print(sorted(df_cleaned['country'].unique()))

# Save the cleaned dataset
df_cleaned.to_csv('HappinesssGlobalData_v5.csv', index=False)

C:\Users\ibrahim\AppData\Local\Temp\ipykernel_7608\1232468271.py:144: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clean = df_clean.groupby(['country', 'year']).apply(merge_country_rows).reset_index(drop=True)


Original number of records: 3221
Cleaned number of records: 2646
Number of unique countries: 189
Number of unique years: 14

Unique countries in cleaned dataset:
['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Central African Republic', 'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Costa Rica', 'Croatia', 'Cuba', 'Cyprus', 'Democratic Republic of Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Ethiopia', 'Fiji', 'Finland', 'France', 'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 'Guatemala', 'Guinea', 'Guinea-Bissau', 

### Drop Columns with more than 30% null values

In [8]:
import pandas as pd

# Load dataset
df = pd.read_csv("HappinesssGlobalData_v5.csv")

# Threshold: only allow <=30% missing data
# → A column must have at least 70% non-null values to be kept
threshold = len(df) * 0.7

dropped_columns = []
kept_columns = []

for col in df.columns:
    if df[col].count() < threshold:
        dropped_columns.append(col)
    else:
        kept_columns.append(col)

# Create cleaned dataset
df_cleaned = df[kept_columns].copy()

print("Dropped columns (more than 30% missing):")
for c in dropped_columns:
    print(" -", c)

print("\nRemaining columns:", len(kept_columns))
print(df_cleaned.head())
# Save cleaned dataset
df_cleaned.to_csv("HappinesssGlobalData_v6.csv", index=False)


Dropped columns (more than 30% missing):
 - happiness_yearly_change
 - poverty_rate
 - poverty_yearly_change
 - poverty_global_percentile
 - extreme_poverty
 - moderate_poverty
 - low_poverty
 - zero_poverty
 - homicide_rate_per_100k
 - male_suicide_rate
 - female_suicide_rate

Remaining columns: 53
       country  year  education_expenditure_pct_gdp  happiness_score  \
0  Afghanistan  2011                        3.46201            4.258   
1  Afghanistan  2012                        2.60420            4.040   
2  Afghanistan  2013                        3.45446              NaN   
3  Afghanistan  2014                        3.69522            3.575   
4  Afghanistan  2015                        3.25580            3.360   

   happiness_global_percentile very_high_happiness very_low_happiness  \
0                    16.167665               False              False   
1                     8.383234               False              False   
2                          NaN                 